# National SOC — depth harmonisation (Jobbágy & Jackson 2000) + RF / XGBoost / QRF
# with random and spatial block nested CV, SHAP, bootstrap maps and QRF prediction-interval maps

Repository: `madagascar-soc-harmonized-mapping` (MIT). Authors: Antsa Sarobidy Randrianantenaina and Nandrianina Ramifehiarivo.

1. **Depth harmonisation**: $SOC_{0-100} = SOC_{0-30} / f_{30}$ with $f_{30} = (1-\beta^{30})/(1-\beta^{100})$, β calibrated per vegetation group on the paired profiles.
2. **Nested CV**, random and spatial blocks: Random Forest (GPU 0), XGBoost (GPU 1), QRF (CPU), in parallel. β re-fitted in every outer training fold; deeper layers validated on measured 1 m profiles; QRF interval coverage reported.
3. Performance figures, random vs spatial, QRF coverage. 3b. Spatial fold maps.
4. SHAP + PDP for the three models.
5. GPU bootstrap maps (RF, XGBoost): mean / SD / CV.
6. **QRF prediction with interval rasters**: mean, q05, q50, q95, 90 % PI width.

Run cells top to bottom; settings in `soc_config.py`.

In [ ]:
!pip install optuna xgboost shap statsmodels rasterio pyproj shapely openpyxl joblib quantile-forest -q
import os; os.chdir('/kaggle/working'); print(os.getcwd())

In [ ]:
%%writefile soc_config.py
# ═══════════════════════════════════════════════════════════════════════════════
#  soc_config.py — single place for paths & settings (edit here only)
# ═══════════════════════════════════════════════════════════════════════════════
import os

WORK = os.environ.get('SOC_WORK', '/kaggle/working')

# ── Inputs ────────────────────────────────────────────────────────────────────
# Full point table with covariates (all 4,184 points; SC30 + SOC100 where measured)
PREDICTORS_CSV = os.environ.get('SOC_PRED_CSV', f'{WORK}/SOC30_predictors_table.csv')
# Hand-corrected rows where SOC100 < SC30 (set to None if not used)
CORRECTIONS_CSV = os.environ.get(
    'SOC_CORR_CSV',
    '/kaggle/input/datasets/antsasarobidyran/deepsoc-data-corrected/'
    'DeepSOC_invalid_negatives_corrected.csv')

# ── Outputs ───────────────────────────────────────────────────────────────────
HARM_DIR = f'{WORK}/Harmonization'
HARM_TABLE = f'{HARM_DIR}/SOC_harmonized_table.csv'
HARM_PARAMS = f'{HARM_DIR}/jobbagy_group_parameters.csv'
RESULTS_DIR = f'{WORK}/ML_SOC_Harmonized'
FIG_DIR = f'{RESULTS_DIR}/Figures'
MAP_DIR = f'{WORK}/Maps_Harmonized'

# ── Harmonisation settings ────────────────────────────────────────────────────
SHRINK_K0 = 10.0         # pseudo-count pulling small groups to the parent / JJ2000
CALIBRATE_ON = 'LU'      # 'LU' = only profiles with a field land-use label
N_BOOT_HARM = 500

# ── Targets (column names produced by the harmonisation step) ─────────────────
TARGETS = {
    'SOC30':   'SC30',
    'DeepSOC': 'DeepSOC30_100_h',
    'SOC100':  'SOC100_h',
}
SOC_UPPER_LIMIT = 1000

# ── Predictors ────────────────────────────────────────────────────────────────
NUMERIC_PREDICTORS = ['NDVI', 'NDWI', 'NIRI', 'elevation', 'slope',
                      'mat', 'map', 'treecover']
CATEGORICAL_PREDICTORS = ['lulc', 'soil_type']

# ── Models ────────────────────────────────────────────────────────────────────
#   RandomForest (cuML, GPU) · XGBoost (GPU) · QRF = quantile regression forest
#   (quantile-forest package, CPU; point prediction = conditional mean)
MODELS = ['RandomForest', 'XGBoost', 'QRF']
QRF_OPTUNA_TRIALS = 30               # QRF is tuned on CPU → fewer trials by default
QRF_QUANTILES = [0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975]
QRF_PI = (0.05, 0.95)                # interval mapped as rasters (90 % PI)

# ── Nested CV (same design as the original notebook) ──────────────────────────
N_OUTER_FOLDS = 10
N_OUTER_REPEATS = 10
N_INNER_FOLDS = 4
OPTUNA_TRIALS = 100

# ── Nested SPATIAL block CV (from soc_spatial_cv.ipynb) ───────────────────────
CV_SCHEMES = ['random', 'spatial']
LON_COL, LAT_COL = 'Longitude', 'Latitude'
SPATIAL_DESIGN = {                   # (block shape, flat-to-flat size in km)
    'SOC30':   ('hexagon', 125),
    'DeepSOC': ('hexagon', 150),     # 150 km from the block-design analysis of
    'SOC100':  ('hexagon', 150),     # the 1,520 profiles measured to 1 m
}
N_SPATIAL_REPEATS = 4                # grid positions (half-block shifts), 1–4
# For DeepSOC / SOC100, balance outer folds on the measured 1 m profiles
# (validation set) rather than on all plots: weight measured = 1, harmonised = w
SPATIAL_HARMONIZED_WEIGHT = 0.1
GEOJSON = os.environ.get('SOC_GEOJSON') or '/kaggle/input/datasets/antsasarobidyran/madaforblockcv/madagascar_ne10m.geojson'
FINAL_MODEL_CV = 'random'            # which tuned final models feed SHAP & maps

# ── Rasters for mapping ───────────────────────────────────────────────────────
_K = '/kaggle/input/datasets/antsasarobidyran'
RASTERS = {
    'NDVI':      f'{_K}/merged-covariates-geotiff-v2/NDVI.tif',
    'NDWI':      f'{_K}/merged-covariates-geotiff-v2/NDWI.tif',
    'NIRI':      f'{_K}/merged-covariates-geotiff-v2/NIRI.tif',
    'elevation': f'{_K}/merged-covariates-geotiff-v6/ELEV_ALOS.tif',
    'lulc':      f'{_K}/merged-covariates-geotiff-v3/lulc.tif',
    'map':       f'{_K}/merged-covariates-geotiff-v3/map_corrected.tif',
    'mat':       f'{_K}/merged-covariates-geotiff-v3/mat_corrected.tif',
    'slope':     f'{_K}/merged-covariates-geotiff-v6/SLOPE_ALOS.tif',
    'soil_type': f'{_K}/merged-covariates-geotiff-v3/soil_type.tif',
    'treecover': f'{_K}/merged-covariates-geotiff-v3/treecover.tif',
}
if os.environ.get('SOC_RASTER_DIR'):      # local testing override
    RASTERS = {k: f"{os.environ['SOC_RASTER_DIR']}/{k}.tif" for k in RASTERS}
MAT_DIVIDE_BY_10 = True                  # raster MAT stored ×10
ZERO_IS_NODATA = ('mat', 'map')

# Raster integer code → class name used in the training table
SOIL_CLASSES = {
    1: 'Andosols', 2: 'Bare Rock', 3: 'Ferrallitic Soils', 4: 'Ferruginous Soils',
    5: 'Fersiallitic Soils', 6: 'Hydromorphic Soils', 7: 'Lithic Raw Mineral Soils',
    8: 'Alluvial Weakly Developed Soils',
    9: 'Erosional Weakly Developed Soils / Lithosols',
    10: 'Podzolic Soils and Podzols', 11: 'Vertisols',
}
LULC_CLASSES = {
    10: 'Tree Cover', 20: 'Shrubland', 30: 'Grassland', 40: 'Cropland',
    50: 'Built-up', 60: 'Bare / Sparse Vegetation', 70: 'Snow and Ice',
    80: 'Permanent Water Bodies', 90: 'Herbaceous Wetland', 95: 'Mangroves',
    100: 'Moss and Lichen',
}
RASTER_CLASS_MAPS = {'lulc': LULC_CLASSES, 'soil_type': SOIL_CLASSES}
# Pixels whose class never occurs in the training data:
#   'mask' → NoData in the map,  'mode' → most frequent training class
UNSEEN_CLASS_POLICY = {'lulc': 'mask', 'soil_type': 'mode'}
MASK_LULC_CODES = (80,)                  # permanent water → always NoData

# ── Quick test mode (SOC_FAST=1): tiny settings for the smoke test ──────────
FAST = os.environ.get('SOC_FAST') == '1'
if FAST:
    N_BOOT_HARM, N_OUTER_FOLDS, N_OUTER_REPEATS = 20, 3, 1
    N_INNER_FOLDS, OPTUNA_TRIALS, N_SPATIAL_REPEATS = 2, 2, 1
    QRF_OPTUNA_TRIALS = 2

# ── Mapping (bootstrap) ───────────────────────────────────────────────────────
N_BOOT_MAP = 3 if FAST else 100

# ── QRF prediction-interval maps (CPU) ────────────────────────────────────────
QRF_MAP_DECIM = 1                    # >1 predicts on a coarser grid (factor)
QRF_MAP_TILE = 2048                  # output tile size (pixels)
QRF_MAP_BATCH = 250_000              # pixels per predict call
TILE_SIZE = 4096
PRED_BATCH = 2_000_000
NODATA_OUT = -9999.0


In [ ]:
%%writefile nature_style.py
# ═══════════════════════════════════════════════════════════════════════════════
#  nature_style.py — shared Nature-Reviews-style figure settings (Arial)
# ═══════════════════════════════════════════════════════════════════════════════
import glob
import os

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

MM = 1 / 25.4                      # mm → inch
SINGLE_COL = 89 * MM               # Nature single column
DOUBLE_COL = 183 * MM              # Nature double column

# Nature Publishing Group palette (ggsci "npg")
NPG = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F',
       '#8491B4', '#91D1C2', '#DC0000', '#7E6148', '#B09C85']
MODEL_COLOR = {'RandomForest': '#3C5488', 'XGBoost': '#E64B35', 'QRF': '#7E6148'}
MODEL_LABEL = {'RandomForest': 'Random forest', 'XGBoost': 'XGBoost', 'QRF': 'QRF'}
MODEL_MARKER = {'RandomForest': 'o', 'XGBoost': 's', 'QRF': '^'}
INK = '#222222'
GREY = '#8C8C8C'
LIGHT = '#D9D9D9'

UNIT = r'Mg C ha$^{-1}$'
DEPTH_ORDER = ['SOC30', 'DeepSOC', 'SOC100']
DEPTH_LABEL = {                       # \mathregular keeps the en dash in Arial
    'SOC30':   r'SOC$_{\mathregular{0–30}}$',
    'DeepSOC': r'SOC$_{\mathregular{30–100}}$',
    'SOC100':  r'SOC$_{\mathregular{0–100}}$',
}

FEATURE_LABEL = {
    'NDVI': 'NDVI', 'NDWI': 'NDWI', 'NIRI': 'NIRI',
    'elevation': 'Elevation (m)', 'slope': 'Slope (°)',
    'mat': 'MAT (°C)', 'map': 'MAP (mm)', 'treecover': 'Tree cover (%)',
    'lulc': 'Land cover', 'soil_type': 'Soil type',
    'lulc_enc': 'Land cover', 'soil_type_enc': 'Soil type',
}


def flabel(k):
    return FEATURE_LABEL.get(k, k)


def setup_arial(verbose=True):
    """Register Arial. Tries (1) system Arial, (2) kagglehub 'hammaadali/arial-font',
    (3) Liberation Sans (metric-compatible clone) as last resort."""
    names = {f.name for f in fm.fontManager.ttflist}
    family = None
    if 'Arial' in names:
        family = 'Arial'
    else:
        try:
            import kagglehub
            p = kagglehub.dataset_download('hammaadali/arial-font')
            ttfs = glob.glob(f'{p}/**/*.ttf', recursive=True)
            for t in ttfs:
                fm.fontManager.addfont(t)
            if ttfs:
                family = fm.FontProperties(fname=ttfs[0]).get_name()
        except Exception as e:                                   # noqa: BLE001
            if verbose:
                print(f'[font] kagglehub Arial unavailable ({e})')
    if family is None:
        family = 'Liberation Sans' if 'Liberation Sans' in names else 'DejaVu Sans'
        if verbose:
            print(f'[font] Arial not found — falling back to {family}')
    apply_style(family)
    if verbose:
        print(f'[font] Active font family → {family}')
    return family


def apply_style(family='Arial'):
    mpl.rcParams.update({
        'font.family': family,
        'font.sans-serif': [family, 'Arial', 'Helvetica', 'Liberation Sans'],
        'mathtext.fontset': 'custom',
        'mathtext.rm': family, 'mathtext.it': f'{family}:italic',
        'mathtext.bf': f'{family}:bold', 'mathtext.sf': family,
        'mathtext.cal': family, 'mathtext.tt': family,
        'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
        'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'legend.fontsize': 6.5,
        'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
        'xtick.minor.width': 0.4, 'ytick.minor.width': 0.4,
        'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
        'xtick.direction': 'out', 'ytick.direction': 'out',
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.edgecolor': INK, 'axes.labelcolor': INK,
        'xtick.color': INK, 'ytick.color': INK, 'text.color': INK,
        'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
        'legend.frameon': False, 'legend.handlelength': 1.2,
        'lines.linewidth': 1.0, 'patch.linewidth': 0.4,
        'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.02, 'savefig.facecolor': 'white',
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
        'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    })


def panel_letter(ax, letter, x=-0.18, y=1.04, size=9):
    """Nature style: bold lower-case letter, no period, top-left."""
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=size,
            fontweight='bold', va='bottom', ha='left', color='black')


def save(fig, path_no_ext, formats=('png', 'pdf')):
    os.makedirs(os.path.dirname(path_no_ext) or '.', exist_ok=True)
    for ext in formats:
        fig.savefig(f'{path_no_ext}.{ext}')
    print(f'Saved → {path_no_ext}.[{",".join(formats)}]')


In [ ]:
%%writefile soc_harmonize.py
# ═══════════════════════════════════════════════════════════════════════════════
#  soc_harmonize.py — depth harmonisation 0–30 cm → 0–100 cm
#  Jobbágy & Jackson (2000) exponential depth model, calibrated on local profiles
# ═══════════════════════════════════════════════════════════════════════════════
#
#  Model (Gale & Grigal 1987; Jobbágy & Jackson 2000, Ecol. Appl. 10:423-436)
#      Y(d) = 1 − β^d                 cumulative fraction of SOC above depth d (cm)
#  Writing β = exp(−k) (k = decay rate, cm⁻¹), the fraction of the 0–100 cm stock
#  held in 0–30 cm is
#      f₃₀ = (1 − β³⁰) / (1 − β¹⁰⁰) = expm1(−30k) / expm1(−100k)
#  so for a profile with only SOC₀₋₃₀:
#      SOC₀₋₁₀₀ = SOC₀₋₃₀ / f₃₀      SOC₃₀₋₁₀₀ = SOC₀₋₁₀₀ − SOC₀₋₃₀
#
#  k (β) is estimated per vegetation group from the paired profiles (SC30 & SOC100
#  both measured). Small groups are shrunk towards their parent group (woody /
#  herbaceous), and parents towards the global Jobbágy & Jackson values
#  (share of 0–100 cm SOC in 0–20 cm: forests 50 %, grasslands 42 %,
#  shrublands 33 %). k < 0 (β > 1) is allowed: SOC increasing with depth
#  (f₃₀ < 0.30), which is common in mangroves and hydromorphic soils.
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from scipy.optimize import brentq

# ── Field land-use label → harmonisation group ────────────────────────────────
LU_TO_GROUP = {
    'Forest': 'Forest', 'Closed Canopy': 'Forest', 'Pine': 'Forest',
    'Reforestation': 'Forest',
    'AgroF': 'Agroforestry', 'AGroF': 'Agroforestry', 'Agroforesterie': 'Agroforestry',
    'Tree fallow': 'Fallow', 'Shrub fallow': 'Fallow', 'Jachere': 'Fallow',
    'Shrubland': 'Shrubland', 'Pseudo-steppe': 'Shrubland',
    'Grassland': 'Grassland', 'Tany Maty': 'Grassland', 'Tany maty': 'Grassland',
    'Bare': 'Grassland',
    'Crop': 'Cropland', 'Cropland': 'Cropland', 'Culture': 'Cropland',
    'Rice': 'Rice/Wetland',
    'Mangroves': 'Mangrove',
}
# ── ESA WorldCover class (fallback when LU is 'undetermined') ─────────────────
ESA_TO_GROUP = {
    'Tree Cover': 'Forest', 'Shrubland': 'Shrubland', 'Grassland': 'Grassland',
    'Cropland': 'Cropland', 'Built-up': 'Grassland',
    'Bare / Sparse Vegetation': 'Grassland', 'Herbaceous Wetland': 'Rice/Wetland',
    'Permanent Water Bodies': 'Rice/Wetland', 'Mangroves': 'Mangrove',
    'Moss and Lichen': 'Grassland', 'Snow and Ice': 'Grassland',
}
PARENT = {
    'Forest': 'Woody', 'Agroforestry': 'Woody', 'Fallow': 'Woody',
    'Shrubland': 'Woody',
    'Grassland': 'Herbaceous', 'Cropland': 'Herbaceous', 'Rice/Wetland': 'Herbaceous',
    'Mangrove': 'Mangrove',
}
# Jobbágy & Jackson (2000) global share of 0–100 cm SOC found in 0–20 cm
JJ2000_F20 = {'Forest': 0.50, 'Grassland': 0.42, 'Shrubland': 0.33}
GROUP_TO_JJ = {
    'Forest': 'Forest', 'Agroforestry': 'Forest', 'Fallow': 'Forest',
    'Woody': 'Forest', 'Shrubland': 'Shrubland',
    'Grassland': 'Grassland', 'Cropland': 'Grassland', 'Rice/Wetland': 'Grassland',
    'Herbaceous': 'Grassland', 'Mangrove': None,
}

K_BOUNDS = (-0.05, 0.50)     # decay rate bounds (cm⁻¹); β ∈ [0.61, 1.05]


# ═══════════════════════════════════════════════════════════════════════════════
#  Depth-model algebra
# ═══════════════════════════════════════════════════════════════════════════════
def frac(k, d_top, d_bot=100.0):
    """Fraction of the 0–d_bot stock contained in 0–d_top for decay rate k."""
    k = np.asarray(k, dtype=float)
    out = np.empty_like(k)
    small = np.abs(k) < 1e-9
    out[small] = d_top / d_bot                         # uniform profile limit
    ks = k[~small]
    out[~small] = np.expm1(-d_top * ks) / np.expm1(-d_bot * ks)
    return out if out.ndim else float(out)


def k_from_ratio(r, d_top=30.0, d_bot=100.0):
    """Invert f(k) = r for one profile (vectorised via loop; ~1 µs each)."""
    r = np.atleast_1d(np.asarray(r, dtype=float))
    lo, hi = K_BOUNDS
    f_lo, f_hi = frac(lo, d_top, d_bot), frac(hi, d_top, d_bot)
    out = np.full(r.shape, np.nan)
    for i, ri in enumerate(r):
        if not np.isfinite(ri):
            continue
        if ri <= f_lo:
            out[i] = lo
        elif ri >= f_hi:
            out[i] = hi
        else:
            out[i] = brentq(lambda kk: frac(kk, d_top, d_bot) - ri, lo, hi, xtol=1e-10)
    return out


def k_from_f20(f20):
    return float(k_from_ratio([f20], d_top=20.0)[0])


JJ2000_K = {g: k_from_f20(v) for g, v in JJ2000_F20.items()}


# ═══════════════════════════════════════════════════════════════════════════════
#  Table preparation
# ═══════════════════════════════════════════════════════════════════════════════
def assign_groups(df, lu_col='LU', esa_col='lulc'):
    """HGROUP from the field LU label; ESA WorldCover class when LU is unknown."""
    g = df[lu_col].astype(str).str.strip().map(LU_TO_GROUP) if lu_col in df else None
    src = pd.Series('LU', index=df.index)
    if g is None:
        g = pd.Series(np.nan, index=df.index, dtype=object)
    miss = g.isna()
    if esa_col in df.columns:
        g.loc[miss] = df.loc[miss, esa_col].astype(str).map(ESA_TO_GROUP)
        src.loc[miss] = 'ESA'
    still = g.isna()
    g.loc[still] = 'Grassland'                     # last-resort default
    src.loc[still] = 'default'
    return g.astype(str), src


def apply_corrections(df, corrected_csv=None, key='num'):
    """Overwrite SC30/SOC100 with the manually corrected values (rows where
    SOC100 < SC30 had been fixed by hand), matched on `key`."""
    df = df.copy()
    if corrected_csv is None:
        return df, 0
    try:
        cor = pd.read_csv(corrected_csv, sep=None, engine='python')
    except FileNotFoundError:
        print(f'[harm] corrections file not found: {corrected_csv} — skipped')
        return df, 0
    if key not in cor.columns:
        print(f"[harm] '{key}' missing in corrections file — skipped")
        return df, 0
    for c in ('SC30', 'SOC100'):
        if c in cor.columns:
            cor[c] = pd.to_numeric(cor[c].astype(str).str.replace(',', '.'),
                                   errors='coerce')
    cor = cor.drop_duplicates(key).set_index(key)
    hit = df[key].isin(cor.index)
    for c in ('SC30', 'SOC100'):
        if c in cor.columns:
            df.loc[hit, c] = df.loc[hit, key].map(cor[c]).values
    return df, int(hit.sum())


def prepare_table(df, lu_col='LU', esa_col='lulc', r_max=0.98, r_min=0.02,
                  calibrate_on='LU'):
    """Adds HGROUP, GROUP_SRC, ratio, k_obs and the calibration flag."""
    df = df.copy()
    df['HGROUP'], df['GROUP_SRC'] = assign_groups(df, lu_col, esa_col)
    df['PARENT'] = df['HGROUP'].map(PARENT)
    meas = df['SOC100'].notna() & df['SC30'].notna() & (df['SOC100'] > 0)
    df['measured100'] = meas
    df['ratio30_100'] = np.where(meas, df['SC30'] / df['SOC100'], np.nan)
    df['k_obs'] = np.nan
    df.loc[meas, 'k_obs'] = k_from_ratio(df.loc[meas, 'ratio30_100'].values)
    calib = meas & df['ratio30_100'].between(r_min, r_max)
    if calibrate_on == 'LU':                       # only profiles with a field LU
        calib &= df['GROUP_SRC'].eq('LU')
    df['calib'] = calib
    return df


# ═══════════════════════════════════════════════════════════════════════════════
#  Calibration (hierarchical shrinkage of the median decay rate)
# ═══════════════════════════════════════════════════════════════════════════════
def fit_params(df_cal, k0=10.0, stat='median'):
    """Returns {group: k} for every group in PARENT (plus parents)."""
    agg = np.median if stat == 'median' else np.mean
    d = df_cal[df_cal['calib']]
    params, info = {}, []
    # parents first: shrink towards Jobbágy & Jackson (2000)
    parent_k = {}
    for p in sorted(set(PARENT.values())):
        sub = d.loc[d['PARENT'] == p, 'k_obs'].dropna()
        jj = GROUP_TO_JJ.get(p)
        prior = JJ2000_K[jj] if jj else (agg(sub) if len(sub) else 0.0)
        n = len(sub)
        kp = (n * agg(sub) + k0 * prior) / (n + k0) if n else prior
        parent_k[p] = kp
        info.append(dict(level='parent', group=p, n=n,
                         k_local=agg(sub) if n else np.nan, k_prior=prior, k=kp))
    for g, p in PARENT.items():
        sub = d.loc[d['HGROUP'] == g, 'k_obs'].dropna()
        n = len(sub)
        kg = (n * agg(sub) + k0 * parent_k[p]) / (n + k0) if n else parent_k[p]
        params[g] = kg
        info.append(dict(level='group', group=g, n=n,
                         k_local=agg(sub) if n else np.nan, k_prior=parent_k[p], k=kg))
    info = pd.DataFrame(info)
    info['beta'] = np.exp(-info['k'])
    info['f30_100'] = frac(info['k'].values, 30.0)
    info['f20_100'] = frac(info['k'].values, 20.0)
    return params, info


def literature_params():
    out = {}
    for g in PARENT:
        jj = GROUP_TO_JJ.get(g)
        out[g] = JJ2000_K[jj] if jj else np.nan
    return out


def harmonize(df, params, keep_measured=True, suffix='_h'):
    """Fill SOC100 / Deep for profiles that only have SC30."""
    df = df.copy()
    k = df['HGROUP'].map(params).astype(float)
    f30 = frac(k.values, 30.0)
    soc100_pred = df['SC30'].values / f30
    df['k_used'] = k.values
    df['f30_100'] = f30
    df['SOC100_jj'] = soc100_pred                      # model value for every row
    if keep_measured:
        use_meas = df['measured100'].values
        df[f'SOC100{suffix}'] = np.where(use_meas, df['SOC100'], soc100_pred)
        df['SOC100_src'] = np.where(use_meas, 'measured', 'harmonized')
    else:
        df[f'SOC100{suffix}'] = soc100_pred
        df['SOC100_src'] = 'harmonized'
    df[f'SOC100{suffix}'] = np.maximum(df[f'SOC100{suffix}'], df['SC30'])
    df[f'DeepSOC30_100{suffix}'] = df[f'SOC100{suffix}'] - df['SC30']
    return df


def bootstrap_uncertainty(df, n_boot=500, k0=10.0, seed=42, suffix='_h'):
    """Bootstrap the calibration profiles → sd / 90 % interval of the
    harmonised SOC100 for every harmonised row (measured rows get 0)."""
    rng = np.random.default_rng(seed)
    cal_idx = np.where(df['calib'].values)[0]
    groups = df['HGROUP'].values
    sc30 = df['SC30'].values
    sims = np.empty((n_boot, len(df)), dtype=np.float32)
    ks = []
    for b in range(n_boot):
        bi = rng.choice(cal_idx, size=len(cal_idx), replace=True)
        boot = df.iloc[bi]
        p, _ = fit_params(boot, k0=k0)
        ks.append(p)
        k = np.array([p[g] for g in groups])
        sims[b] = sc30 / frac(k, 30.0)
    sd = sims.std(0)
    lo, hi = np.percentile(sims, [5, 95], axis=0)
    harm = df['SOC100_src'].values == 'harmonized'
    df = df.copy()
    df[f'SOC100{suffix}_sd'] = np.where(harm, sd, 0.0)
    df[f'SOC100{suffix}_p05'] = np.where(harm, lo, df[f'SOC100{suffix}'])
    df[f'SOC100{suffix}_p95'] = np.where(harm, hi, df[f'SOC100{suffix}'])
    return df, pd.DataFrame(ks)


# ═══════════════════════════════════════════════════════════════════════════════
#  Validation on the paired profiles (repeated K-fold)
# ═══════════════════════════════════════════════════════════════════════════════
def _metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    if len(y) < 3:
        return dict(n=len(y), r2=np.nan, rmse=np.nan, mae=np.nan, bias=np.nan, ccc=np.nan)
    res = y - p
    r2 = 1 - np.sum(res ** 2) / np.sum((y - y.mean()) ** 2)
    ccc = 2 * np.cov(y, p, ddof=0)[0, 1] / (y.var() + p.var() + (y.mean() - p.mean()) ** 2)
    return dict(n=len(y), r2=r2, rmse=np.sqrt(np.mean(res ** 2)),
                mae=np.mean(np.abs(res)), bias=np.mean(p - y), ccc=ccc)


def validate(df, n_splits=10, n_repeats=10, k0=10.0, seed=42):
    """Out-of-fold SOC100 for every calibration profile under 3 schemes:
    'local' (group-calibrated JJ), 'literature' (global JJ2000 β),
    'single' (one β for all profiles)."""
    from sklearn.model_selection import KFold
    cal = df[df['calib']].copy()
    idx = np.arange(len(cal))
    oof = {s: np.zeros(len(cal)) for s in ('local', 'single')}
    cnt = np.zeros(len(cal))
    for rep in range(n_repeats):
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed + rep)
        for tr, te in kf.split(idx):
            p, _ = fit_params(cal.iloc[tr], k0=k0)
            k_te = cal.iloc[te]['HGROUP'].map(p).values
            oof['local'][te] += cal.iloc[te]['SC30'].values / frac(k_te, 30.0)
            k_single = np.median(cal.iloc[tr]['k_obs'])
            oof['single'][te] += cal.iloc[te]['SC30'].values / frac(
                np.full(len(te), k_single), 30.0)
            cnt[te] += 1
    for s in oof:
        oof[s] /= np.maximum(cnt, 1)
    lit = literature_params()
    k_lit = cal['HGROUP'].map(lit).values
    oof['literature'] = cal['SC30'].values / frac(k_lit, 30.0)
    out = cal[['HGROUP', 'SC30', 'SOC100']].copy()
    for s, v in oof.items():
        out[f'SOC100_{s}'] = v
    rows = []
    for subset, mask in [('All profiles', np.ones(len(out), bool)),
                         ('Excl. mangroves', out['HGROUP'].ne('Mangrove').values)]:
        for s in ('local', 'literature', 'single'):
            for tgt in ('SOC100', 'DeepSOC'):
                y = out['SOC100'].values[mask]
                p = out[f'SOC100_{s}'].values[mask]
                if tgt == 'DeepSOC':
                    y = y - out['SC30'].values[mask]
                    p = p - out['SC30'].values[mask]
                rows.append(dict(subset=subset, scheme=s, target=tgt, **_metrics(y, p)))
    for g in sorted(out['HGROUP'].unique()):
        m = out['HGROUP'].eq(g).values
        rows.append(dict(subset=g, scheme='local', target='SOC100',
                         **_metrics(out['SOC100'].values[m], out['SOC100_local'].values[m])))
    return out, pd.DataFrame(rows)


def run_harmonization(df, k0=10.0, calibrate_on='LU', n_boot=500, seed=42):
    """Convenience: prepare → fit → harmonise → bootstrap. Returns (table, info)."""
    t = prepare_table(df, calibrate_on=calibrate_on)
    params, info = fit_params(t, k0=k0)
    t = harmonize(t, params)
    if n_boot:
        t, _ = bootstrap_uncertainty(t, n_boot=n_boot, k0=k0, seed=seed)
    return t, params, info


In [ ]:
%%writefile soc_spatial.py
# ═══════════════════════════════════════════════════════════════════════════════
#  soc_spatial.py — hexagonal spatial blocks for nested spatial CV
#  (same grid / fold logic as soc_spatial_cv.ipynb: UTM 38S, fixed origin,
#   4 repeats = grid shifted by half a block, greedy balanced fold assignment)
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
from scipy.spatial import cKDTree

ORIGIN = np.array([300.0, 7100.0])          # grid origin, km, UTM 38S


def lonlat_to_km(lon, lat):
    from pyproj import Transformer
    t = Transformer.from_crs(4326, 32738, always_xy=True)
    x, y = t.transform(np.asarray(lon, float), np.asarray(lat, float))
    return np.c_[x, y] / 1000.0


def grid_shifts(size):
    return [(0.0, 0.0), (size / 2, 0.0), (0.0, size / 2), (size / 2, size / 2)]


def block_keys(xy, shape_, size, shift=(0.0, 0.0)):
    p = xy - ORIGIN - np.asarray(shift)
    if shape_ == 'square':
        return np.floor(p / size).astype(int)
    # pointy-top hexagons, flat-to-flat width = size (cube-coordinate rounding)
    r = size / np.sqrt(3)
    q = (np.sqrt(3) / 3 * p[:, 0] - p[:, 1] / 3) / r
    z = (2 / 3 * p[:, 1]) / r
    y = -q - z
    rx, ry, rz = np.round(q), np.round(y), np.round(z)
    dx, dy, dz = abs(rx - q), abs(ry - y), abs(rz - z)
    fx = (dx > dy) & (dx > dz)
    fz = ~fx & ~((dy > dz) & ~fx)
    rx = np.where(fx, -ry - rz, rx)
    rz = np.where(fz, -rx - ry, rz)
    return np.c_[rx, rz].astype(int)


def assign_folds(keys, k, weights=None):
    """Largest block first, always into the fold with the lowest load.
    `weights` (per plot) lets the load count e.g. measured 1 m profiles only."""
    blocks, inv = np.unique(keys, axis=0, return_inverse=True)
    inv = inv.ravel()
    if len(blocks) < k:
        return None, len(blocks)
    w = np.ones(len(keys)) if weights is None else np.asarray(weights, float)
    load_b = np.bincount(inv, weights=w, minlength=len(blocks))
    cnt_b = np.bincount(inv, minlength=len(blocks))
    load = np.zeros(k)
    fold_of_block = np.empty(len(blocks), int)
    for j in sorted(range(len(blocks)), key=lambda j: (-load_b[j], -cnt_b[j], j)):
        f = int(np.argmin(load))
        fold_of_block[j] = f
        load[f] += load_b[j]
    return fold_of_block[inv] + 1, len(blocks)


def build_folds(xy, shape_, size, k, shift=(0.0, 0.0), weights=None):
    # shrink the blocks only if there are too few of them to fill k folds
    while size >= 5:
        keys = block_keys(xy, shape_, size, shift)
        ids, n_blocks = assign_folds(keys, k, weights)
        if ids is not None and len(np.unique(ids)) == k:
            return dict(ids=ids, keys=keys, size=size, n_blocks=n_blocks)
        size *= 0.85
    raise RuntimeError('cannot build spatial folds')


def spatial_splits(xy, shape_, size, k, shift=(0.0, 0.0), weights=None):
    folds = build_folds(xy, shape_, size, k, shift, weights)
    splits = [(np.flatnonzero(folds['ids'] != f), np.flatnonzero(folds['ids'] == f))
              for f in range(1, k + 1)]
    return splits, folds


def test_train_dist(xy, fold):
    d = np.empty(len(xy))
    for f in np.unique(fold):
        test = fold == f
        d[test] = cKDTree(xy[~test]).query(xy[test])[0]
    return d


In [ ]:
%%writefile soc_ml.py
# ═══════════════════════════════════════════════════════════════════════════════
#  soc_ml.py — data loading, encoders, GPU model factory, fold-wise harmonisation
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

import soc_config as C
import soc_harmonize as H


# ── GPU availability ──────────────────────────────────────────────────────────
def gpu_available():
    try:
        import cupy as cp
        return cp.cuda.runtime.getDeviceCount() > 0
    except Exception:                                            # noqa: BLE001
        return False


# ── Data ──────────────────────────────────────────────────────────────────────
def load_harmonized_table(path=C.HARM_TABLE):
    df = pd.read_csv(path).replace(-9999, np.nan)
    df = df[df['SC30'].notna() & (df['SC30'] < C.SOC_UPPER_LIMIT)]
    df = df[~(df['SOC100'] >= C.SOC_UPPER_LIMIT)].reset_index(drop=True)
    for col in C.NUMERIC_PREDICTORS:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())
    encoders = {}
    for col in C.CATEGORICAL_PREDICTORS:
        df[col] = df[col].astype(str).fillna('Unknown')
        le = LabelEncoder().fit(df[col])
        df[f'{col}_enc'] = le.transform(df[col])
        encoders[col] = le
    features = C.NUMERIC_PREDICTORS + [f'{c}_enc' for c in C.CATEGORICAL_PREDICTORS]
    df['calib'] = df['calib'].astype(bool)
    df['measured100'] = df['measured100'].astype(bool)
    return df, features, encoders


def targets_for_fold(df, target, train_idx, test_idx, k0=C.SHRINK_K0):
    """Target vectors with β re-fitted on the training fold only (no leakage).
    Returns y_train, y_test_all (harmonised with train β), y_test_meas (NaN
    where the profile was not measured to 1 m), and the fold's group params."""
    if target == 'SOC30':
        y = df['SC30'].values
        return y[train_idx], y[test_idx], y[test_idx], None
    params, _ = H.fit_params(df.iloc[train_idx], k0=k0)
    h = H.harmonize(df, params)
    col = C.TARGETS[target]
    y_all = h[col].values
    meas = df['measured100'].values
    if target == 'SOC100':
        y_meas = np.where(meas, df['SOC100'].values, np.nan)
    else:
        y_meas = np.where(meas, np.maximum(df['SOC100'].values - df['SC30'].values, 0),
                          np.nan)
    return y_all[train_idx], y_all[test_idx], y_meas[test_idx], params


def full_target(df, target, k0=C.SHRINK_K0):
    if target == 'SOC30':
        return df['SC30'].values, None
    params, _ = H.fit_params(df, k0=k0)
    return H.harmonize(df, params)[C.TARGETS[target]].values, params


# ── Model factory ─────────────────────────────────────────────────────────────
def make_model(model_name, params, use_gpu=True, seed=42):
    p = dict(params)
    if model_name == 'QRF':
        use_gpu = False
    if model_name == 'RandomForest':
        if use_gpu:
            from cuml.ensemble import RandomForestRegressor as cuRF
            return cuRF(**p, random_state=seed, n_streams=1)
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(**p, random_state=seed, n_jobs=-1)
    if model_name == 'QRF':
        from quantile_forest import RandomForestQuantileRegressor
        return RandomForestQuantileRegressor(**p, max_samples_leaf=None,
                                             default_quantiles='mean',
                                             random_state=seed, n_jobs=-1)
    if model_name == 'XGBoost':
        import xgboost as xgb
        return xgb.XGBRegressor(**p, random_state=seed, verbosity=0,
                                tree_method='hist',
                                device='cuda' if use_gpu else 'cpu')
    raise ValueError(model_name)


def to_sklearn_rf(params, seed=42):
    """Same hyper-parameters on sklearn (needed by shap.TreeExplainer)."""
    from sklearn.ensemble import RandomForestRegressor
    return RandomForestRegressor(**params, random_state=seed, n_jobs=-1)


def is_qrf(model):
    return type(model).__name__ == 'RandomForestQuantileRegressor'


def predict(model, X):
    X = np.ascontiguousarray(X, dtype=np.float32)
    if is_qrf(model):
        return np.asarray(model.predict(X, quantiles='mean'), dtype=np.float64).ravel()
    out = model.predict(X)
    try:
        import cupy as cp
        if isinstance(out, cp.ndarray):
            out = cp.asnumpy(out)
    except ImportError:
        pass
    if hasattr(out, 'to_numpy'):
        out = out.to_numpy()
    return np.asarray(out, dtype=np.float64).ravel()


def predict_quantiles(model, X, quantiles):
    """QRF conditional quantiles, shape (n, len(quantiles)), monotone per row."""
    q = model.predict(np.ascontiguousarray(X, dtype=np.float32), quantiles=list(quantiles))
    q = np.asarray(q, dtype=np.float64).reshape(len(X), -1)
    return np.maximum.accumulate(q, axis=1)


def interval_scores(y, lo, hi):
    """PICP (coverage) and MPIW (mean width) of [lo, hi]."""
    y, lo, hi = (np.asarray(a, float) for a in (y, lo, hi))
    m = np.isfinite(y)
    if m.sum() == 0:
        return np.nan, np.nan
    return float(np.mean((y[m] >= lo[m]) & (y[m] <= hi[m]))), float(np.mean(hi[m] - lo[m]))


def suggest_params(trial, model_name):
    if model_name in ('RandomForest', 'QRF'):
        return dict(
            n_estimators=trial.suggest_int('n_estimators', 100, 1000),
            max_depth=trial.suggest_int('max_depth', 5, 24),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 15),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
            max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', 1.0]),
        )
    return dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 1000),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 0.0, 2.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.5, 5.0),
    )


def calc_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]
    keys = ('rmse', 'r2', 'mae', 'bias', 'rpd', 'rpiq', 'ccc')
    if len(y_true) < 3:
        return dict(n=len(y_true), **{k: np.nan for k in keys})
    res = y_true - y_pred
    rmse = float(np.sqrt(np.mean(res ** 2)))
    sst = np.sum((y_true - y_true.mean()) ** 2)
    r2 = float(1 - np.sum(res ** 2) / sst) if sst > 0 else np.nan
    iqr = np.percentile(y_true, 75) - np.percentile(y_true, 25)
    ccc_den = y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2
    ccc = 2 * np.cov(y_true, y_pred, ddof=0)[0, 1] / ccc_den if ccc_den > 0 else np.nan
    return dict(n=int(len(y_true)), rmse=rmse, r2=r2, mae=float(np.mean(np.abs(res))),
                bias=float(np.mean(y_pred - y_true)),
                rpd=float(y_true.std() / rmse) if rmse > 0 else np.nan,
                rpiq=float(iqr / rmse) if rmse > 0 else np.nan, ccc=float(ccc))


def results_dir(target, cv='random'):
    import os
    return os.path.join(C.RESULTS_DIR, f'results_{target}' + ('_spatialCV' if cv == 'spatial' else ''))


In [ ]:
%%writefile soc_raster.py
# ═══════════════════════════════════════════════════════════════════════════════
#  soc_raster.py — covariate raster stack: aligned tiled reading, categorical
#  look-up tables, masks and pixel areas (shared by the two mapping scripts)
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.windows import Window

import soc_config as C


def build_cat_lut(col, encoder, mode_idx=None):
    """raster integer code → training-encoder index; -1 means masked (NoData).
    Codes are translated through their class *name* (SOIL/LULC_CLASSES)."""
    cmap = C.RASTER_CLASS_MAPS[col]
    classes = list(encoder.classes_)
    lut = np.full(max(cmap) + 2, -1, dtype=np.int32)
    policy = C.UNSEEN_CLASS_POLICY.get(col, 'mask')
    unseen = []
    for code, name in cmap.items():
        if name in classes:
            lut[code] = classes.index(name)
        else:
            unseen.append(name)
            lut[code] = mode_idx if (policy == 'mode' and mode_idx is not None) else -1
    if col == 'lulc':
        for code in C.MASK_LULC_CODES:
            if code < len(lut):
                lut[code] = -1
    print(f'   LUT {col}: {len(cmap) - len(unseen)} classes matched; '
          f'unseen {unseen} → {policy}')
    return lut


def pixel_area_ha(transform, crs, rows):
    """Area (ha) of the pixels lying in the given rows of a grid."""
    if crs is not None and crs.is_geographic:
        R = 6371008.8
        lat_top = transform.f + transform.e * rows
        lat_bot = lat_top + transform.e
        dlon = np.deg2rad(abs(transform.a))
        a = R ** 2 * dlon * np.abs(np.sin(np.deg2rad(lat_top)) - np.sin(np.deg2rad(lat_bot)))
        return a / 1e4
    return np.full(len(rows), abs(transform.a * transform.e) / 1e4)


class RasterStack:
    """All covariates opened once. Optional decimation `decim` (>1 predicts on a
    grid `decim` times coarser: continuous layers averaged, categorical nearest)."""

    def __init__(self, encoders, modes, tile=C.TILE_SIZE, decim=1):
        self.h = {k: rasterio.open(v) for k, v in C.RASTERS.items()}
        ref = self.h['NDVI'] if 'NDVI' in self.h else next(iter(self.h.values()))
        for k, src in self.h.items():
            if (src.height, src.width) != (ref.height, ref.width) or src.transform != ref.transform:
                raise ValueError(f'Raster {k} is not aligned with the reference grid')
        self.d = int(max(1, decim))
        self.src_H, self.src_W = ref.height, ref.width
        self.H, self.W = ref.height // self.d, ref.width // self.d
        self.transform = ref.transform * Affine.scale(self.d)
        self.crs = ref.crs
        self.tile = tile
        self.luts = {c: build_cat_lut(c, encoders[c], modes.get(c))
                     for c in C.CATEGORICAL_PREDICTORS}
        prof = ref.profile.copy()
        prof.update(dtype='float32', count=1, nodata=C.NODATA_OUT, compress='lzw',
                    predictor=2, tiled=True, blockxsize=512, blockysize=512,
                    BIGTIFF='YES', width=self.W, height=self.H, transform=self.transform)
        self.profile = prof
        self.tiles = [(r, c) for r in range(0, self.H, tile) for c in range(0, self.W, tile)]

    def window(self, r0, c0):
        return Window(c0, r0, min(self.tile, self.W - c0), min(self.tile, self.H - r0))

    def read(self, r0, c0):
        """Feature matrix of the valid pixels of an output tile.
        Returns X (n, p) float32, flat indices into the tile, output window."""
        win = self.window(r0, c0)
        th, tw = int(win.height), int(win.width)
        src_win = Window(win.col_off * self.d, win.row_off * self.d, tw * self.d, th * self.d)
        valid = np.ones((th, tw), bool)
        cols = {}
        for f in C.NUMERIC_PREDICTORS + C.CATEGORICAL_PREDICTORS:
            src = self.h[f]
            is_cat = f in C.CATEGORICAL_PREDICTORS
            if self.d == 1:
                arr = src.read(1, window=src_win).astype(np.float32)
            else:
                arr = src.read(1, window=src_win, out_shape=(th, tw), masked=True,
                               resampling=Resampling.nearest if is_cat else Resampling.average)
                arr = np.ma.filled(arr.astype(np.float32), np.nan)
            bad = ~np.isfinite(arr)
            if src.nodata is not None:
                bad |= np.isclose(arr, src.nodata)
            if f in C.ZERO_IS_NODATA:
                bad |= arr == 0
            if is_cat:
                lut = self.luts[f]
                code = np.clip(np.where(bad, 0, arr).astype(np.int64), 0, len(lut) - 1)
                enc = lut[code]
                bad |= enc < 0
                arr = enc.astype(np.float32)
            elif f == 'mat' and C.MAT_DIVIDE_BY_10:
                arr = arr / 10.0
            valid &= ~bad
            cols[f] = arr
        flat = np.flatnonzero(valid)
        if flat.size == 0:
            return None, None, win
        order = C.NUMERIC_PREDICTORS + C.CATEGORICAL_PREDICTORS
        X = np.column_stack([cols[f].ravel()[flat] for f in order]).astype(np.float32)
        return X, flat, win

    def close(self):
        for src in self.h.values():
            src.close()


In [ ]:
%%writefile train_soc_harm.py
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
#  train_soc_harm.py — repeated nested CV (GPU) on the depth-harmonised table
#
#  python train_soc_harm.py --target SOC100 --model XGBoost --cv random
#  python train_soc_harm.py --target SOC100 --model XGBoost --cv spatial
#  (the launcher sets CUDA_VISIBLE_DEVICES so each process sees one GPU)
#
#  --cv random  : outer 10 × 10 KFold, inner 4-fold KFold (Optuna TPE + pruning)
#  --cv spatial : outer 10 folds of hexagonal blocks × 4 grid positions,
#                 inner 4 spatial-block folds built from the outer-training plots
#                 only (same design as soc_spatial_cv.ipynb)
#  • For DeepSOC / SOC100 the Jobbágy β are re-fitted on every outer-training
#    fold, so held-out profiles never inform their own harmonised target.
#  • Validation is reported on (i) measured 0–100 cm profiles only (primary,
#    independent) and (ii) all held-out points (measured + harmonised).
#  • --model QRF (quantile regression forest, CPU): the conditional quantiles of
#    every held-out point are also stored, and the coverage (PICP) and mean width
#    (MPIW) of the 50/80/90/95 % central prediction intervals are reported.
# ═══════════════════════════════════════════════════════════════════════════════
import argparse
import json
import os
import time

import joblib
import numpy as np
import optuna
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.model_selection import KFold

import soc_config as C
import soc_ml as M
import soc_spatial as S

optuna.logging.set_verbosity(optuna.logging.WARNING)


def inner_search(model_name, X, y, splits, n_trials, use_gpu, seed=42):
    """Optuna search; `splits` = list of (train, val) index arrays into X."""
    def objective(trial):
        params = M.suggest_params(trial, model_name)
        scores = []
        for i, (tr, va) in enumerate(splits):
            mdl = M.make_model(model_name, params, use_gpu)
            mdl.fit(X[tr], y[tr])
            p = M.predict(mdl, X[va])
            scores.append(float(np.sqrt(np.mean((y[va] - p) ** 2))))
            trial.report(float(np.mean(scores)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=seed),
                                pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value


def random_kfold(n, k, seed):
    return list(KFold(n_splits=k, shuffle=True, random_state=seed).split(np.arange(n)))


def outer_plan(a, df, xy):
    """Yields (repeat, fold, train_idx, test_idx, inner_splits_factory, info)."""
    n = len(df)
    if a.cv == 'random':
        for rep in range(a.repeats):
            for fold, (tr, te) in enumerate(random_kfold(n, a.outer_folds, rep * 100 + 42), 1):
                yield rep + 1, fold, tr, te, (lambda tr_=tr: random_kfold(len(tr_), a.inner_folds, 0)), {}
        return
    shape_, size = C.SPATIAL_DESIGN[a.target]
    w = None
    if a.target != 'SOC30':                    # balance folds on measured 1 m profiles
        w = np.where(df['measured100'].values, 1.0, C.SPATIAL_HARMONIZED_WEIGHT)
    for rep, shift in enumerate(S.grid_shifts(size)[:a.spatial_repeats], 1):
        outer, folds = S.spatial_splits(xy, shape_, size, a.outer_folds, shift, w)
        sizes = np.bincount(folds['ids'])[1:]
        n_meas = [int(df['measured100'].values[te].sum()) for _, te in outer]
        print(f'  repeat {rep}: shift {shift} km, block {folds["size"]:.0f} km, '
              f'{folds["n_blocks"]} blocks, fold sizes {sizes.min()}–{sizes.max()}, '
              f'measured-1m per fold {min(n_meas)}–{max(n_meas)}', flush=True)
        yield ('folds', rep, folds['ids'])
        for fold, (tr, te) in enumerate(outer, 1):
            def inner(tr_=tr, shift_=shift):
                sp, inf = S.spatial_splits(xy[tr_], shape_, size, a.inner_folds, shift_,
                                           None if w is None else w[tr_])
                return sp
            yield rep, fold, tr, te, inner, dict(block_km=folds['size'])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--target', required=True, choices=list(C.TARGETS))
    ap.add_argument('--model', required=True, choices=C.MODELS)
    ap.add_argument('--cv', default='random', choices=['random', 'spatial'])
    ap.add_argument('--outer-folds', type=int, default=C.N_OUTER_FOLDS)
    ap.add_argument('--repeats', type=int, default=C.N_OUTER_REPEATS)
    ap.add_argument('--spatial-repeats', type=int, default=C.N_SPATIAL_REPEATS)
    ap.add_argument('--inner-folds', type=int, default=C.N_INNER_FOLDS)
    ap.add_argument('--trials', type=int, default=None,
                    help=f'Optuna trials (default {C.OPTUNA_TRIALS}; QRF {C.QRF_OPTUNA_TRIALS})')
    ap.add_argument('--cpu', action='store_true', help='force CPU (testing)')
    a = ap.parse_args()

    if a.trials is None:
        a.trials = C.QRF_OPTUNA_TRIALS if a.model == 'QRF' else C.OPTUNA_TRIALS
    use_gpu = (not a.cpu) and M.gpu_available() and a.model != 'QRF'
    is_qrf = a.model == 'QRF'
    QS = C.QRF_QUANTILES
    PIS = {50: (0.25, 0.75), 80: (0.10, 0.90), 90: (0.05, 0.95), 95: (0.025, 0.975)}
    out_dir = M.results_dir(a.target, a.cv)
    os.makedirs(os.path.join(out_dir, 'models'), exist_ok=True)
    tag = f'{a.model}|{a.target}|{a.cv}'
    n_rep = a.repeats if a.cv == 'random' else a.spatial_repeats
    print('=' * 78)
    print(f' {a.model} | {a.target} | {a.cv.upper()} CV | GPU={use_gpu} | '
          f'outer {a.outer_folds}×{n_rep}, inner {a.inner_folds}, {a.trials} trials')
    if a.cv == 'spatial':
        print(f' blocks: {C.SPATIAL_DESIGN[a.target][0]}s {C.SPATIAL_DESIGN[a.target][1]} km')
    print('=' * 78, flush=True)

    df, features, encoders = M.load_harmonized_table()
    X = df[features].values.astype(np.float32)
    n = len(df)
    xy = S.lonlat_to_km(df[C.LON_COL].values, df[C.LAT_COL].values)
    src = df['SOC100_src'].values if a.target != 'SOC30' else np.array(['measured'] * n)
    print(f' n = {n}  | measured 0–100 cm = {int(df.measured100.sum())}', flush=True)

    pred_rows, fold_rows, param_rows, fold_table = [], [], [], {}
    t0 = time.time()
    run, total = 0, a.outer_folds * n_rep
    for item in outer_plan(a, df, xy):
        if item[0] == 'folds':
            fold_table[f'fold_rep{item[1]}'] = item[2]
            continue
        rep, fold, tr, te, inner_factory, info = item
        run += 1
        y_tr, y_te_all, y_te_meas, _ = M.targets_for_fold(df, a.target, tr, te)
        best, best_rmse = inner_search(a.model, X[tr], y_tr, inner_factory(),
                                       a.trials, use_gpu, seed=42 + run)
        mdl = M.make_model(a.model, best, use_gpu, seed=42 + run)
        mdl.fit(X[tr], y_tr)
        p = M.predict(mdl, X[te])
        dist = cKDTree(xy[tr]).query(xy[te])[0]

        m_meas = M.calc_metrics(y_te_meas, p)
        m_all = M.calc_metrics(y_te_all, p)
        qcols, pi_stats = {}, {}
        if is_qrf:
            qp = M.predict_quantiles(mdl, X[te], QS)
            qcols = {f'q{q:.3f}': qp[:, i] for i, q in enumerate(QS)}
            lo, hi = qp[:, QS.index(0.05)], qp[:, QS.index(0.95)]
            pi_stats['meas_picp90'], pi_stats['meas_mpiw90'] = M.interval_scores(y_te_meas, lo, hi)
            pi_stats['all_picp90'], pi_stats['all_mpiw90'] = M.interval_scores(y_te_all, lo, hi)
        fold_rows.append(dict(repeat=rep, fold=fold, n_test=len(te), **pi_stats,
                              median_test_train_km=float(np.median(dist)),
                              **{f'meas_{k}': v for k, v in m_meas.items()},
                              **{f'all_{k}': v for k, v in m_all.items()}))
        param_rows.append(dict(repeat=rep, fold=fold, inner_rmse=best_rmse, **info, **best))
        pred_rows.append(pd.DataFrame(dict(
            row_id=te, repeat=rep, fold=fold, predicted=p,
            observed_all=y_te_all, observed_meas=y_te_meas, source=src[te],
            HGROUP=df['HGROUP'].values[te], x_km=xy[te, 0], y_km=xy[te, 1],
            dist_to_train_km=dist, **qcols)))
        el = (time.time() - t0) / 60
        print(f' [{tag}] fold {run:3d}/{total}  n={len(te):4d}  '
              f'dist={np.median(dist):6.1f} km  R²(meas)={m_meas["r2"]:.3f}  '
              f'R²(all)={m_all["r2"]:.3f}  RMSE(meas)={m_meas["rmse"]:.1f}  | '
              f'{el:.1f} min, ETA {el / run * (total - run):.1f} min', flush=True)

    preds = pd.concat(pred_rows, ignore_index=True)
    folds = pd.DataFrame(fold_rows)
    pooled_meas = M.calc_metrics(preds['observed_meas'], preds['predicted'])
    pooled_all = M.calc_metrics(preds['observed_all'], preds['predicted'])

    # QRF: coverage of central prediction intervals (pooled over all outer folds)
    qrf_cov = []
    if is_qrf:
        for lev, (ql, qh) in PIS.items():
            lo, hi = preds[f'q{ql:.3f}'].values, preds[f'q{qh:.3f}'].values
            for scope, col in [('measured', 'observed_meas'), ('all', 'observed_all')]:
                picp, mpiw = M.interval_scores(preds[col].values, lo, hi)
                qrf_cov.append(dict(nominal=lev, scope=scope, picp=100 * picp, mpiw=mpiw))
        pd.DataFrame(qrf_cov).to_csv(os.path.join(out_dir, 'qrf_interval_coverage.csv'),
                                     index=False)
        print(pd.DataFrame(qrf_cov).round(2).to_string(index=False))

    # OOF mean per sample (averaged over repeats) → used for scatter plots
    oof = (preds.groupby('row_id')
           .agg(predicted=('predicted', 'mean'), pred_sd=('predicted', 'std'),
                observed_all=('observed_all', 'mean'),
                observed_meas=('observed_meas', 'first'),
                source=('source', 'first'), HGROUP=('HGROUP', 'first'),
                dist_to_train_km=('dist_to_train_km', 'median'))
           .reset_index())

    # ── final model on all data (tuned with the same CV scheme) ───────────────
    y_full, harm_params = M.full_target(df, a.target)
    if a.cv == 'random':
        full_splits = random_kfold(n, a.inner_folds, 0)
    else:
        w = None if a.target == 'SOC30' else np.where(
            df['measured100'].values, 1.0, C.SPATIAL_HARMONIZED_WEIGHT)
        full_splits, _ = S.spatial_splits(xy, *C.SPATIAL_DESIGN[a.target],
                                          a.inner_folds, weights=w)
    best, best_rmse = inner_search(a.model, X, y_full, full_splits, a.trials, use_gpu)
    final = M.make_model(a.model, best, use_gpu)
    final.fit(X, y_full)

    bundle = dict(model=final, best_params=best, features=features,
                  encoders=encoders, target=a.target, cv=a.cv,
                  target_col=C.TARGETS[a.target], harm_params=harm_params,
                  gpu=use_gpu)
    joblib.dump(bundle, os.path.join(out_dir, 'models', f'{a.model}.pkl'))
    if a.model == 'XGBoost':
        final.save_model(os.path.join(out_dir, 'models', 'XGBoost.ubj'))

    preds.to_csv(os.path.join(out_dir, f'cv_predictions_{a.model}.csv'), index=False)
    oof.to_csv(os.path.join(out_dir, f'oof_mean_{a.model}.csv'), index=False)
    folds.to_csv(os.path.join(out_dir, f'fold_metrics_{a.model}.csv'), index=False)
    pd.DataFrame(param_rows).to_csv(
        os.path.join(out_dir, f'best_params_per_fold_{a.model}.csv'), index=False)
    if fold_table:                              # spatial folds actually used
        ft = pd.concat([df[[C.LON_COL, C.LAT_COL, 'SC30', 'measured100']],
                        pd.DataFrame(xy, columns=['x_km', 'y_km']),
                        pd.DataFrame(fold_table)], axis=1)
        ft.to_csv(os.path.join(out_dir, 'spatial_folds.csv'), index=False)

    def _mean_sd(prefix):
        ks = ('r2', 'rmse', 'mae', 'bias', 'rpd', 'rpiq', 'ccc')
        return {f'{prefix}{k}_mean': float(np.nanmean(folds[f'{prefix}{k}'])) for k in ks} | \
               {f'{prefix}{k}_sd': float(np.nanstd(folds[f'{prefix}{k}'])) for k in ks}

    summary = dict(model=a.model, target=a.target, cv_scheme=a.cv, n=n,
                   n_measured_100=int(df.measured100.sum()),
                   cv=f'{n_rep}x{a.outer_folds} outer, {a.inner_folds} inner, '
                      f'{a.trials} trials', gpu=use_gpu,
                   design=(f'{C.SPATIAL_DESIGN[a.target][0]} '
                           f'{C.SPATIAL_DESIGN[a.target][1]} km' if a.cv == 'spatial' else 'random'),
                   median_test_train_km=float(preds['dist_to_train_km'].median()),
                   pooled_measured=pooled_meas, pooled_all=pooled_all,
                   **_mean_sd('meas_'), **_mean_sd('all_'),
                   final_params={k: (v.item() if isinstance(v, np.generic) else v)
                                 for k, v in best.items()},
                   final_inner_rmse=best_rmse, qrf_coverage=qrf_cov,
                   minutes=(time.time() - t0) / 60)
    with open(os.path.join(out_dir, f'summary_{a.model}.json'), 'w') as f:
        json.dump(summary, f, indent=2, default=float)
    print(f'\n POOLED (measured): R²={pooled_meas["r2"]:.3f} RMSE={pooled_meas["rmse"]:.2f} '
          f'CCC={pooled_meas["ccc"]:.3f} n={pooled_meas["n"]}')
    print(f' POOLED (all)     : R²={pooled_all["r2"]:.3f} RMSE={pooled_all["rmse"]:.2f} '
          f'CCC={pooled_all["ccc"]:.3f} n={pooled_all["n"]}')
    print(f' median test→train distance: {summary["median_test_train_km"]:.1f} km')
    print(f' ✅ saved → {out_dir}  ({summary["minutes"]:.1f} min)', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile map_soc_gpu.py
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
#  map_soc_gpu.py — bootstrap ensemble mapping on GPU (mean, SD, CV rasters)
#
#  python map_soc_gpu.py --target SOC100 --model RandomForest [--n-boot 100]
#
#  Phase 1 (GPU): B bootstrap replicates of the training table. For DeepSOC /
#    SOC100 the Jobbágy β are RE-ESTIMATED on each bootstrap sample, so the
#    uncertainty of the depth harmonisation propagates into the map SD.
#    RF → cuML RandomForestRegressor   XGBoost → device='cuda'
#  Phase 2 (GPU): tiles → feature matrix → predictions of every replicate,
#    Welford running mean/variance → mean, SD and CV (%) GeoTIFFs + national
#    stock total (Pg C).
#
#  Categorical rasters are mapped raster code → class name → training-encoder
#  index (soc_raster.build_cat_lut).
#  QRF prediction intervals are mapped separately by map_qrf_intervals.py.
# ═══════════════════════════════════════════════════════════════════════════════
import argparse
import gc
import json
import math
import os
import pickle
import time

import numpy as np
import rasterio

import soc_config as C
import soc_harmonize as H
import soc_ml as M
from soc_raster import RasterStack, pixel_area_ha


# ── helpers ───────────────────────────────────────────────────────────────────
def gpu_predict(model, model_name, X, use_gpu):
    if not use_gpu:
        return M.predict(model, X).astype(np.float32)
    import cupy as cp
    out = np.empty(len(X), np.float32)
    for s in range(0, len(X), C.PRED_BATCH):
        Xg = cp.asarray(X[s:s + C.PRED_BATCH])
        if model_name == 'XGBoost':
            p = model.get_booster().inplace_predict(Xg)
        else:
            p = model.predict(Xg)
        out[s:s + C.PRED_BATCH] = cp.asnumpy(cp.asarray(p).ravel())
        del Xg, p
    cp.get_default_memory_pool().free_all_blocks()
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  Phase 1 — bootstrap training
# ═══════════════════════════════════════════════════════════════════════════════
def phase1(a, out_dir, use_gpu):
    bundle = __import__('joblib').load(
        os.path.join(M.results_dir(a.target, a.cv), 'models', f'{a.model}.pkl'))
    print(f'   hyper-parameters from {a.cv}-CV tuning: {bundle["best_params"]}')
    params = bundle['best_params']
    df, features, encoders = M.load_harmonized_table()
    X = df[features].values.astype(np.float32)
    n = len(df)
    mdir = os.path.join(out_dir, 'boot_models')
    os.makedirs(mdir, exist_ok=True)
    rng = np.random.RandomState(42)
    y_ref, _ = M.full_target(df, a.target)          # reference for OOB diagnostics
    modes = {c: int(df[f'{c}_enc'].mode()[0]) for c in C.CATEGORICAL_PREDICTORS}
    diag = []
    t0 = time.time()
    for b in range(a.n_boot):
        f = os.path.join(mdir, f'b{b:03d}.pkl')
        bi = rng.randint(0, n, n)                       # always draw → reproducible
        if os.path.exists(f):
            continue
        boot = df.iloc[bi].reset_index(drop=True)
        if a.target == 'SOC30':
            y = boot['SC30'].values
            k_used = {}
        else:
            p_b, _ = H.fit_params(boot, k0=C.SHRINK_K0)    # β re-estimated
            y = H.harmonize(boot, p_b)[C.TARGETS[a.target]].values
            k_used = p_b
        mdl = M.make_model(a.model, params, use_gpu, seed=1000 + b)
        mdl.fit(X[bi], y)
        oob = np.setdiff1d(np.arange(n), np.unique(bi))
        m = M.calc_metrics(y_ref[oob], gpu_predict(mdl, a.model, X[oob], use_gpu))
        diag.append(dict(b=b, oob_r2=m['r2'], oob_rmse=m['rmse'], **{f'k_{g}': v for g, v in k_used.items()}))
        with open(f, 'wb') as fh:
            pickle.dump(mdl, fh)
        del mdl
        gc.collect()
        print(f'   boot {b + 1:3d}/{a.n_boot}  OOB R²={m["r2"]:.3f}  '
              f'({(time.time() - t0) / 60:.1f} min)', flush=True)
    if diag:
        import pandas as pd
        pd.DataFrame(diag).to_csv(os.path.join(out_dir, 'bootstrap_diagnostics.csv'),
                                  index=False, mode='a',
                                  header=not os.path.exists(
                                      os.path.join(out_dir, 'bootstrap_diagnostics.csv')))
    return features, encoders, modes


# ═══════════════════════════════════════════════════════════════════════════════
#  Phase 2 — tiled GPU prediction (batch-outer / tile-inner, Welford)
# ═══════════════════════════════════════════════════════════════════════════════
def phase2(a, out_dir, features, encoders, modes, use_gpu):
    stack = RasterStack(encoders, modes, tile=C.TILE_SIZE)
    tiles = stack.tiles
    print(f'   grid {stack.H}×{stack.W}, {len(tiles)} tiles, CRS {stack.crs}')
    build_X = stack.read
    prof = stack.profile

    mdir = os.path.join(out_dir, 'boot_models')
    files = sorted(os.listdir(mdir))[:a.n_boot]
    batch = a.batch_models or (len(files) if a.model == 'XGBoost' else 10)
    n_batches = math.ceil(len(files) / batch)
    acc = {}                                           # tile → [mean, m2, k, flat, win]
    t0 = time.time()
    for bi in range(n_batches):
        fb = files[bi * batch:(bi + 1) * batch]
        models = [pickle.load(open(os.path.join(mdir, f), 'rb')) for f in fb]
        for ti, (r0, c0) in enumerate(tiles):
            X, flat, win = build_X(r0, c0)
            if X is None:
                continue
            if ti not in acc:
                acc[ti] = [np.zeros(len(X), np.float32), np.zeros(len(X), np.float32),
                           0, flat, win]
            mean, m2, k = acc[ti][0], acc[ti][1], acc[ti][2]
            for mdl in models:
                p = np.maximum(gpu_predict(mdl, a.model, X, use_gpu), 0)
                k += 1
                d = p - mean
                mean += d / k
                m2 += d * (p - mean)
            acc[ti][2] = k
            del X
        del models
        gc.collect()
        el = (time.time() - t0) / 60
        print(f'   batch {bi + 1}/{n_batches} done | {el:.1f} min | '
              f'ETA {el / (bi + 1) * (n_batches - bi - 1):.1f} min', flush=True)

    paths = {s: os.path.join(out_dir, f'{a.target}_{a.model}_{s}.tif')
             for s in ('mean', 'sd', 'cv')}
    total_pg, total_sd_pg, area_tot = 0.0, 0.0, 0.0
    with rasterio.open(paths['mean'], 'w', **prof) as fm, \
            rasterio.open(paths['sd'], 'w', **prof) as fs, \
            rasterio.open(paths['cv'], 'w', **prof) as fc:
        for ti, (r0, c0) in enumerate(tiles):
            win = stack.window(r0, c0)
            mt = np.full((win.height, win.width), C.NODATA_OUT, np.float32)
            st, ct = mt.copy(), mt.copy()
            if ti in acc:
                mean, m2, k, flat, _ = acc[ti]
                sd = np.sqrt(np.maximum(m2 / max(k - 1, 1), 0))
                rr, cc = np.unravel_index(flat, (win.height, win.width))
                mt[rr, cc], st[rr, cc] = mean, sd
                ct[rr, cc] = np.where(mean > 0, 100 * sd / np.maximum(mean, 1e-6), C.NODATA_OUT)
                area = pixel_area_ha(stack.transform, stack.crs, r0 + rr)
                total_pg += float(np.sum(mean * area)) / 1e9
                total_sd_pg += float(np.sum(sd * area)) / 1e9   # upper bound (full correlation)
                area_tot += float(area.sum())
            fm.write(mt, 1, window=win)
            fs.write(st, 1, window=win)
            fc.write(ct, 1, window=win)
    stack.close()
    stats = dict(target=a.target, model=a.model, n_models=len(files),
                 total_stock_PgC=total_pg, total_sd_PgC_upper=total_sd_pg,
                 mapped_area_Mha=area_tot / 1e6,
                 mean_density_MgC_ha=total_pg * 1e9 / max(area_tot, 1),
                 minutes=(time.time() - t0) / 60)
    json.dump(stats, open(os.path.join(out_dir, 'map_stats.json'), 'w'), indent=2)
    print(f'   ✅ {paths["mean"]}\n   total stock = {total_pg:.3f} Pg C '
          f'over {area_tot / 1e6:.2f} Mha', flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--target', required=True, choices=list(C.TARGETS))
    ap.add_argument('--model', required=True, choices=['RandomForest', 'XGBoost'])
    ap.add_argument('--n-boot', type=int, default=C.N_BOOT_MAP)
    ap.add_argument('--batch-models', type=int, default=0,
                    help='models held on the GPU at once (0 = auto)')
    ap.add_argument('--cv', default=C.FINAL_MODEL_CV, choices=['random', 'spatial'],
                    help='which tuned final model to bootstrap')
    ap.add_argument('--cpu', action='store_true')
    a = ap.parse_args()
    use_gpu = (not a.cpu) and M.gpu_available()
    out_dir = os.path.join(C.MAP_DIR, f'{a.target}_{a.model}')
    os.makedirs(out_dir, exist_ok=True)
    print('=' * 70)
    print(f' MAP {a.model} | {a.target} | B={a.n_boot} | GPU={use_gpu}')
    print('=' * 70, flush=True)
    features, encoders, modes = phase1(a, out_dir, use_gpu)
    phase2(a, out_dir, features, encoders, modes, use_gpu)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile map_qrf_intervals.py
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
#  map_qrf_intervals.py — QRF prediction-interval rasters (CPU)
#
#  python map_qrf_intervals.py --target SOC100 [--decim 1] [--cv random]
#
#  Uses the final QRF tuned in the nested CV (results_<target>[_spatialCV]/models/
#  QRF.pkl) and predicts, tile by tile, for every valid pixel:
#     mean      conditional mean (point map)
#     q05, q50, q95   conditional quantiles (C.QRF_PI gives the interval bounds)
#     piw90     q95 − q05             (width of the 90 % prediction interval)
#     rpiw90    100 · piw90 / q50 (%) (relative width)
#  plus national totals. Pixels are masked exactly like in map_soc_gpu.py.
# ═══════════════════════════════════════════════════════════════════════════════
import argparse
import json
import os
import time

import joblib
import numpy as np
import rasterio

import soc_config as C
import soc_ml as M
from soc_raster import RasterStack, pixel_area_ha


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--target', required=True, choices=list(C.TARGETS))
    ap.add_argument('--cv', default=C.FINAL_MODEL_CV, choices=['random', 'spatial'])
    ap.add_argument('--decim', type=int, default=C.QRF_MAP_DECIM,
                    help='predict on a grid N times coarser (1 = native)')
    a = ap.parse_args()

    lo_q, hi_q = C.QRF_PI
    level = int(round(100 * (hi_q - lo_q)))
    names = ['mean', f'q{100 * lo_q:02.0f}', 'q50', f'q{100 * hi_q:02.0f}',
             f'piw{level}', f'rpiw{level}']
    bundle = joblib.load(os.path.join(M.results_dir(a.target, a.cv), 'models', 'QRF.pkl'))
    model, encoders = bundle['model'], bundle['encoders']
    df, _, _ = M.load_harmonized_table()
    modes = {c: int(df[f'{c}_enc'].mode()[0]) for c in C.CATEGORICAL_PREDICTORS}

    out_dir = os.path.join(C.MAP_DIR, f'{a.target}_QRF')
    os.makedirs(out_dir, exist_ok=True)
    print('=' * 70)
    print(f' QRF prediction intervals | {a.target} | {level} % PI [{lo_q}, {hi_q}] | '
          f'decim={a.decim} | params from {a.cv} CV')
    print('=' * 70, flush=True)

    stack = RasterStack(encoders, modes, tile=C.QRF_MAP_TILE, decim=a.decim)
    print(f'   output grid {stack.H}×{stack.W}, {len(stack.tiles)} tiles', flush=True)
    paths = {n: os.path.join(out_dir, f'{a.target}_QRF_{n}.tif') for n in names}
    dsts = {n: rasterio.open(p, 'w', **stack.profile) for n, p in paths.items()}
    tot = dict(mean=0.0, lo=0.0, hi=0.0, area=0.0, n=0)
    t0 = time.time()
    for ti, (r0, c0) in enumerate(stack.tiles):
        X, flat, win = stack.read(r0, c0)
        th, tw = int(win.height), int(win.width)
        layers = {n: np.full((th, tw), C.NODATA_OUT, np.float32) for n in names}
        if X is not None:
            mean = np.empty(len(X), np.float32)
            q = np.empty((len(X), 3), np.float32)
            for s in range(0, len(X), C.QRF_MAP_BATCH):
                xb = X[s:s + C.QRF_MAP_BATCH]
                mean[s:s + len(xb)] = M.predict(model, xb)
                q[s:s + len(xb)] = M.predict_quantiles(model, xb, [lo_q, 0.5, hi_q])
            mean, q = np.maximum(mean, 0), np.maximum(q, 0)
            width = q[:, 2] - q[:, 0]
            rel = np.where(q[:, 1] > 0, 100 * width / np.maximum(q[:, 1], 1e-6), C.NODATA_OUT)
            rr, cc = np.unravel_index(flat, (th, tw))
            for n, v in zip(names, [mean, q[:, 0], q[:, 1], q[:, 2], width, rel]):
                layers[n][rr, cc] = v
            area = pixel_area_ha(stack.transform, stack.crs, r0 + rr)
            tot['mean'] += float(np.sum(mean * area)) / 1e9
            tot['lo'] += float(np.sum(q[:, 0] * area)) / 1e9
            tot['hi'] += float(np.sum(q[:, 2] * area)) / 1e9
            tot['area'] += float(area.sum())
            tot['n'] += len(X)
        for n in names:
            dsts[n].write(layers[n], 1, window=win)
        el = (time.time() - t0) / 60
        print(f'   tile {ti + 1}/{len(stack.tiles)}  valid={0 if X is None else len(X):,}  '
              f'{el:.1f} min, ETA {el / (ti + 1) * (len(stack.tiles) - ti - 1):.1f} min',
              flush=True)
    for d in dsts.values():
        d.close()
    stack.close()
    stats = dict(target=a.target, model='QRF', cv_params=a.cv, decim=a.decim,
                 interval=[lo_q, hi_q], n_pixels=tot['n'],
                 total_stock_PgC=tot['mean'],
                 total_stock_PgC_sum_of_lower=tot['lo'],
                 total_stock_PgC_sum_of_upper=tot['hi'],
                 note='sums of pixel quantiles bound the total; they are not '
                      'quantiles of the national total',
                 mapped_area_Mha=tot['area'] / 1e6,
                 minutes=(time.time() - t0) / 60)
    json.dump(stats, open(os.path.join(out_dir, 'map_stats.json'), 'w'), indent=2)
    print(f'   ✅ {out_dir}\n   total stock (mean) = {tot["mean"]:.3f} Pg C '
          f'[{tot["lo"]:.3f}, {tot["hi"]:.3f}] over {tot["area"] / 1e6:.2f} Mha', flush=True)


if __name__ == '__main__':
    main()


## Step 1 — Depth harmonisation (Jobbágy) + validation + Fig. S1

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — DEPTH HARMONISATION (Jobbágy & Jackson 2000, locally calibrated)
#  → SOC_harmonized_table.csv  +  validation tables  +  Fig. S1 (Nature style)
# ═══════════════════════════════════════════════════════════════════════════════
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import nature_style as NS
import soc_config as C
import soc_harmonize as H

FONT = NS.setup_arial()
os.makedirs(C.HARM_DIR, exist_ok=True)

# ── 1. Load full point table & apply the hand corrections ─────────────────────
raw = pd.read_csv(C.PREDICTORS_CSV).replace(-9999, np.nan)
raw = raw.rename(columns={'SOC100_ten': 'SOC100', 'SOC0_30': 'SC30', 'SOC0_100': 'SOC100'})
raw, n_cor = H.apply_corrections(raw, C.CORRECTIONS_CSV)
print(f'Points: {len(raw)} | SC30: {raw.SC30.notna().sum()} | '
      f'SOC100 measured: {raw.SOC100.notna().sum()} | corrected rows applied: {n_cor}')
bad = (raw.SOC100 < raw.SC30).sum()
print(f'Profiles still with SOC100 < SC30: {bad} → kept as measured (Deep = 0), '
      f'excluded from β calibration')

# ── 2. Groups, per-profile decay rate, calibration ───────────────────────────
tab = H.prepare_table(raw, calibrate_on=C.CALIBRATE_ON)
params, info = H.fit_params(tab, k0=C.SHRINK_K0)
tab = H.harmonize(tab, params)
tab, boot_k = H.bootstrap_uncertainty(tab, n_boot=C.N_BOOT_HARM, k0=C.SHRINK_K0)

# bootstrap CI for every group's β / f30
for g in info['group']:
    if g in boot_k:
        kb = boot_k[g].values
        info.loc[info.group == g, 'f30_p05'] = np.percentile(H.frac(kb, 30.), 5)
        info.loc[info.group == g, 'f30_p95'] = np.percentile(H.frac(kb, 30.), 95)
info['n_harmonized'] = info['group'].map(
    tab[tab.SOC100_src == 'harmonized'].groupby('HGROUP').size()).fillna(0).astype(int)
info['JJ2000_f30'] = info['group'].map(
    lambda g: H.frac(H.JJ2000_K[H.GROUP_TO_JJ[g]], 30.) if H.GROUP_TO_JJ.get(g) else np.nan)
info.to_csv(C.HARM_PARAMS, index=False)
print('\nJobbágy depth-model parameters (k: cm⁻¹, β = e^-k, f30 = share of 0–100 cm in 0–30 cm)')
print(info.round(4).to_string(index=False))

# ── 3. Validation on paired profiles (10 × 10-fold) ──────────────────────────
oof_h, val = H.validate(tab, n_splits=10, n_repeats=2 if C.FAST else 10, k0=C.SHRINK_K0)
val.to_csv(f'{C.HARM_DIR}/harmonization_validation.csv', index=False)
oof_h.to_csv(f'{C.HARM_DIR}/harmonization_oof.csv', index=False)
print('\nValidation (out-of-fold, paired profiles)')
print(val.round(3).to_string(index=False))

# ── 4. Save harmonised table ──────────────────────────────────────────────────
tab.to_csv(C.HARM_TABLE, index=False)
print(f'\n✅ Harmonised table → {C.HARM_TABLE}')
print(tab.groupby(['HGROUP', 'SOC100_src']).agg(
    n=('SC30', 'size'), SC30=('SC30', 'mean'), SOC100=('SOC100_h', 'mean'),
    Deep=('DeepSOC30_100_h', 'mean'), sd=('SOC100_h_sd', 'mean')).round(1).to_string())
print('\nGroup source for harmonised points:',
      tab[tab.SOC100_src == 'harmonized'].GROUP_SRC.value_counts().to_dict())

# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. S1 — harmonisation (183 mm, 2 × 2)
# ═══════════════════════════════════════════════════════════════════════════════
ORDER = ['Forest', 'Agroforestry', 'Fallow', 'Shrubland', 'Grassland',
         'Cropland', 'Rice/Wetland', 'Mangrove']
GCOL = dict(zip(ORDER, NS.NPG[:7] + ['#7E6148']))
gi = info[info.level == 'group'].set_index('group')

fig, axs = plt.subplots(2, 2, figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.78),
                        gridspec_kw=dict(wspace=0.38, hspace=0.45))

# a — observed f30 per group + calibrated value + JJ2000
ax = axs[0, 0]
cal = tab[tab.calib]
rng = np.random.default_rng(1)
for i, g in enumerate(ORDER):
    v = cal.loc[cal.HGROUP == g, 'ratio30_100'].values
    if len(v):
        ax.scatter(i + rng.uniform(-0.22, 0.22, len(v)), v, s=2.5, color=GCOL[g],
                   alpha=0.35, lw=0, rasterized=True)
        ax.boxplot(v, positions=[i], widths=0.55, showfliers=False,
                   medianprops=dict(color=NS.INK, lw=0.8),
                   boxprops=dict(lw=0.6), whiskerprops=dict(lw=0.6),
                   capprops=dict(lw=0.6))
    f, lo, hi = gi.loc[g, ['f30_100', 'f30_p05', 'f30_p95']]
    ax.errorbar(i + 0.38, f, yerr=[[f - lo], [hi - f]], fmt='D', ms=3,
                color='black', elinewidth=0.6, capsize=1.5, zorder=5)
    jj = gi.loc[g, 'JJ2000_f30']
    if np.isfinite(jj):
        ax.plot([i - 0.3, i + 0.3], [jj, jj], color=NS.GREY, ls='--', lw=0.8, zorder=4)
ax.axhline(0.30, color=NS.LIGHT, lw=0.6, zorder=0)
ax.text(-0.45, 0.285, 'uniform profile', fontsize=5.5, color=NS.GREY,
        ha='left', va='top')
ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels([f'{g}\n(n={int(gi.loc[g, "n"])})' for g in ORDER],
                   rotation=45, ha='right', rotation_mode='anchor', fontsize=5.8)
ax.set_ylabel(r'SOC$_{\mathregular{0–30}}$ / SOC$_{\mathregular{0–100}}$')
ax.set_ylim(0, 1.18)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.plot([], [], 'D', color='black', ms=3, label='Calibrated (90% CI)')
ax.plot([], [], '--', color=NS.GREY, lw=0.8, label='Jobbágy & Jackson (2000)')
ax.legend(loc='upper center', fontsize=5.5, ncol=2, columnspacing=1.0)
NS.panel_letter(ax, 'a', x=-0.16)

# b — cumulative SOC distribution with depth
ax = axs[0, 1]
d = np.linspace(0, 100, 201)
for g in ORDER:
    k = gi.loc[g, 'k']
    y = np.array([H.frac(k, di) if di > 0 else 0.0 for di in d])
    ax.plot(y, d, color=GCOL[g], lw=1.0, label=g)
for lab, kk in H.JJ2000_K.items():
    y = np.array([H.frac(kk, di) if di > 0 else 0.0 for di in d])
    ax.plot(y, d, color=NS.GREY, lw=0.6, ls='--')
ax.plot([0, 1], [0, 100], color=NS.LIGHT, lw=0.6, zorder=0)
ax.axhline(30, color=NS.INK, lw=0.4, ls=':')
ax.set_ylim(100, 0)
ax.set_xlim(0, 1)
ax.set_xlabel('Cumulative fraction of SOC$_{\\mathregular{0–100}}$')
ax.set_ylabel('Depth (cm)')
ax.plot([], [], color=NS.GREY, lw=0.6, ls='--', label='JJ2000 (F, G, S)')
ax.legend(fontsize=5.5, loc='lower left', handlelength=1.0)
NS.panel_letter(ax, 'b', x=-0.16)


def dens_scatter(ax, x, y, cmap='viridis'):
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    try:
        z = gaussian_kde(np.vstack([x, y]))(np.vstack([x, y]))
    except Exception:                                            # noqa: BLE001
        z = np.ones_like(x)
    o = z.argsort()
    ax.scatter(x[o], y[o], c=z[o], s=3, cmap=cmap, lw=0, rasterized=True)
    lim = [0, max(x.max(), y.max()) * 1.05]
    ax.plot(lim, lim, color=NS.INK, lw=0.6, ls='--')
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect('equal')


# c — OOF validation (non-mangrove profiles, local calibration)
ax = axs[1, 0]
nm = oof_h[oof_h.HGROUP != 'Mangrove']
dens_scatter(ax, nm.SOC100.values, nm.SOC100_local.values)
r = val[(val.subset == 'Excl. mangroves') & (val.target == 'SOC100')].set_index('scheme')
ax.text(0.04, 0.97,
        f"Local calibration\n$R^2$ = {r.loc['local', 'r2']:.2f}\n"
        f"RMSE = {r.loc['local', 'rmse']:.1f}\nCCC = {r.loc['local', 'ccc']:.2f}\n"
        f"JJ2000 $R^2$ = {r.loc['literature', 'r2']:.2f}\n$n$ = {int(r.loc['local', 'n'])}",
        transform=ax.transAxes, va='top', fontsize=5.8, linespacing=1.35)
ax.set_xlabel(f'Measured SOC$_{{\\mathregular{{0–100}}}}$ ({NS.UNIT})')
ax.set_ylabel(f'Harmonised SOC$_{{\\mathregular{{0–100}}}}$ ({NS.UNIT})')
NS.panel_letter(ax, 'c', x=-0.16)

# d — measured vs harmonised distributions per group
ax = axs[1, 1]
pos, ticks = 0, []
for g in [g for g in ORDER if g != 'Mangrove']:
    for j, (s, colr) in enumerate([('measured', NS.GREY), ('harmonized', GCOL[g])]):
        v = tab.loc[(tab.HGROUP == g) & (tab.SOC100_src == s), 'SOC100_h'].values
        if len(v) > 2:
            vp = ax.violinplot(v, positions=[pos + j * 0.42], widths=0.4,
                               showextrema=False, showmedians=False)
            for b in vp['bodies']:
                b.set_facecolor(colr)
                b.set_edgecolor('none')
                b.set_alpha(0.8)
            ax.plot(pos + j * 0.42, np.median(v), 'o', ms=1.8, color='white',
                    mec=NS.INK, mew=0.4)
    ticks.append(pos + 0.21)
    pos += 1.2
ax.set_xticks(ticks)
ax.set_xticklabels([g for g in ORDER if g != 'Mangrove'], rotation=45, ha='right',
                   rotation_mode='anchor', fontsize=5.8)
ax.set_ylabel(f'SOC$_{{\\mathregular{{0–100}}}}$ ({NS.UNIT})')
ax.fill_between([], [], color=NS.GREY, label='Measured')
ax.fill_between([], [], color=NS.NPG[0], label='Harmonised')
ax.legend(fontsize=5.5, loc='upper right')
NS.panel_letter(ax, 'd', x=-0.16)

NS.save(fig, f'{C.HARM_DIR}/FigS1_depth_harmonization')
plt.show()


## Step 2 — Nested CV: RF (GPU 0), XGBoost (GPU 1), QRF (CPU) × random & spatial CV
To fit Kaggle's 12 h limit, run one scheme or target per session (`CV_TO_RUN`, `TARGETS_TO_RUN`) or pass `EXTRA_ARGS = ['--repeats','5','--trials','50']`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — LAUNCH NESTED CV  (RF → GPU 0, XGBoost → GPU 1, QRF → CPU, in parallel)
#  Each model runs, on its own GPU, every CV scheme × target sequentially:
#    random  : 10 × 10-fold KFold outer, 4-fold inner
#    spatial : 10 hexagon-block folds × 4 grid shifts outer, 4 block folds inner
#  Separate processes avoid the "cudaErrorInitializationError" seen with fork.
# ═══════════════════════════════════════════════════════════════════════════════
import os
import subprocess
import sys
import time

import soc_config as C

TARGETS_TO_RUN = ['SOC30', 'DeepSOC', 'SOC100']     # split across sessions if needed
CV_TO_RUN = C.CV_SCHEMES                             # ['random', 'spatial']
QUEUES = {0: 'RandomForest', 1: 'XGBoost', 'cpu': 'QRF'}   # gpu id (or 'cpu') → model
QUEUES = {g: m for g, m in QUEUES.items() if m in C.MODELS}
EXTRA_ARGS = []        # e.g. ['--repeats', '5', '--trials', '50'] for a faster run

n_gpu = 0
try:
    import cupy as cp
    n_gpu = cp.cuda.runtime.getDeviceCount()
except Exception:                                                # noqa: BLE001
    pass
print(f'GPUs visible: {n_gpu}')

SRC = os.path.dirname(os.path.abspath(C.__file__))        # folder holding the modules
log_dir = os.path.join(C.RESULTS_DIR, 'logs')
os.makedirs(log_dir, exist_ok=True)
procs = {}
for gpu, model in QUEUES.items():
    cmds = ' ; '.join(           # ';' → a failed job does not stop the queue
        f'{sys.executable} -u {SRC}/train_soc_harm.py --target {t} --model {model} --cv {cv} '
        + ' '.join(EXTRA_ARGS) for cv in CV_TO_RUN for t in TARGETS_TO_RUN)
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = '' if gpu == 'cpu' else str(gpu if n_gpu > 1 else 0)
    env['PYTHONPATH'] = SRC + os.pathsep + env.get('PYTHONPATH', '')
    log = open(os.path.join(log_dir, f'log_{model}.txt'), 'w')
    procs[model] = (subprocess.Popen(cmds, shell=True, env=env, stdout=log,
                                     stderr=subprocess.STDOUT, cwd=SRC), log)
    print(f'Launched {model} on {"CPU" if gpu == "cpu" else "GPU " + env["CUDA_VISIBLE_DEVICES"]} → {log.name}')

# ── live monitor (last line of each log every 60 s) ──────────────────────────
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(60)
    for model, (p, log) in procs.items():
        try:
            with open(log.name) as f:
                lines = [l.strip() for l in f if l.strip()]
            print(f'[{model:<12}] {"running" if p.poll() is None else "done"} | '
                  f'{lines[-1] if lines else ""}', flush=True)
        except FileNotFoundError:
            pass
for model, (p, log) in procs.items():
    log.close()
    print(f'{model}: exit code {p.returncode}')


## Step 3 — Performance table, observed vs predicted, random vs spatial, QRF interval coverage

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — MODEL SUMMARY TABLE + OBSERVED-vs-PREDICTED SCATTERPLOTS
#  for both CV schemes (random KFold and spatial block CV)
#  Fig. 2a/2b : RF / XGBoost × 3 depths, measured profiles — random / spatial CV
#  Fig. S2a/b : same, all held-out points (measured + harmonised)
#  Fig. 2c    : random vs spatial CV comparison
# ═══════════════════════════════════════════════════════════════════════════════
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

import nature_style as NS
import soc_config as C
import soc_ml as M

FONT = NS.setup_arial(verbose=False)
MODELS = C.MODELS
LETTERS = 'abcdefghijklmnopqr'

# ── summary table ─────────────────────────────────────────────────────────────
rows = []
for cvs, d, m in [(cvs, d, m) for cvs in C.CV_SCHEMES for d in NS.DEPTH_ORDER for m in MODELS]:
        p = os.path.join(M.results_dir(d, cvs), f'summary_{m}.json')
        if not os.path.exists(p):
            continue
        s = json.load(open(p))
        for scope, key in [('measured', 'meas_'), ('all', 'all_')]:
            if d == 'SOC30' and scope == 'all':
                continue
            pooled = s['pooled_measured' if scope == 'measured' else 'pooled_all']
            rows.append({'CV': cvs, 'Depth': d, 'Model': m, 'Validation': scope,
                         'n': pooled['n'],
                         'Median_test_train_km': round(s.get('median_test_train_km', np.nan), 1),
                         'R2': f"{s[f'{key}r2_mean']:.3f} ± {s[f'{key}r2_sd']:.3f}",
                         'RMSE': f"{s[f'{key}rmse_mean']:.1f} ± {s[f'{key}rmse_sd']:.1f}",
                         'MAE': f"{s[f'{key}mae_mean']:.1f} ± {s[f'{key}mae_sd']:.1f}",
                         'Bias': f"{s[f'{key}bias_mean']:.2f} ± {s[f'{key}bias_sd']:.2f}",
                         'CCC': f"{s[f'{key}ccc_mean']:.3f} ± {s[f'{key}ccc_sd']:.3f}",
                         'RPIQ': f"{s[f'{key}rpiq_mean']:.2f} ± {s[f'{key}rpiq_sd']:.2f}",
                         'Pooled_R2': round(pooled['r2'], 3),
                         'Pooled_RMSE': round(pooled['rmse'], 2)})
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(C.RESULTS_DIR, 'Table_model_performance.csv'), index=False)
print(summary.to_string(index=False))


def load_oof(depth, model, cvs):
    return pd.read_csv(os.path.join(M.results_dir(depth, cvs), f'oof_mean_{model}.csv'))


def density(x, y):
    try:
        z = gaussian_kde(np.vstack([x, y]))(np.vstack([x, y]))
    except Exception:                                            # noqa: BLE001
        z = np.ones_like(x)
    return (z - z.min()) / (np.ptp(z) + 1e-12)


def panel(ax, obs, pred, model, depth, letter, extra=None, cmap='viridis'):
    dz = density(obs, pred)
    o = dz.argsort()
    ax.scatter(obs[o], pred[o], c=dz[o], cmap=cmap, s=2.2,
               lw=0, alpha=0.9, rasterized=True, zorder=2)
    if extra is not None:                         # harmonised points (Fig. S2)
        ax.scatter(extra[0], extra[1], s=1.5, color=NS.GREY, lw=0, alpha=0.35,
                   rasterized=True, zorder=1)
    hi = np.nanpercentile(np.r_[obs, pred], 99.7) * 1.08
    ax.plot([0, hi], [0, hi], color=NS.INK, lw=0.6, ls='--', zorder=3)
    z = np.polyfit(obs, pred, 1)
    xx = np.linspace(0, hi, 50)
    ax.plot(xx, np.polyval(z, xx), color=NS.MODEL_COLOR[model], lw=1.0, zorder=4)
    ax.set_xlim(0, hi)
    ax.set_ylim(0, hi)
    ax.set_aspect('equal')
    ax.locator_params(nbins=4)
    mt = (M.calc_metrics(obs, pred) if extra is None else
          M.calc_metrics(np.r_[obs, extra[0]], np.r_[pred, extra[1]]))
    ax.text(0.04, 0.97,
            f"$R^2$ = {mt['r2']:.2f}\nRMSE = {mt['rmse']:.1f}\nMAE = {mt['mae']:.1f}\n"
            f"Bias = {mt['bias']:.1f}\nCCC = {mt['ccc']:.2f}\n$n$ = {mt['n']:,}",
            transform=ax.transAxes, va='top', ha='left', fontsize=5.6,
            linespacing=1.3, zorder=5,
            bbox=dict(fc='white', ec='none', alpha=0.7, pad=0.6))
    ax.set_title(f'{NS.MODEL_LABEL[model]} · {NS.DEPTH_LABEL[depth]}', fontsize=6.5, pad=3)
    NS.panel_letter(ax, letter, x=-0.30, y=1.02)


def make_figure(scope, cvs, fname):
    nr = len(MODELS)
    fig, axs = plt.subplots(nr, 3, figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.33 * nr),
                            squeeze=False,
                            gridspec_kw=dict(wspace=0.42, hspace=0.38))
    k = 0
    for r, m in enumerate(MODELS):
        for c, d in enumerate(NS.DEPTH_ORDER):
            ax = axs[r, c]
            try:
                oof = load_oof(d, m, cvs)
            except FileNotFoundError:
                ax.set_axis_off()
                continue
            extra = None
            if scope == 'measured' or d == 'SOC30':
                s = oof[oof.observed_meas.notna()]
                obs, pred = s.observed_meas.values, s.predicted.values
            else:
                s = oof[oof.source == 'measured']
                h = oof[oof.source == 'harmonized']
                obs, pred = s.observed_meas.values, s.predicted.values
                extra = (h.observed_all.values, h.predicted.values)
            panel(ax, obs, pred, m, d, LETTERS[k], extra)
            if r == nr - 1:
                ax.set_xlabel(f'Observed ({NS.UNIT})')
            if c == 0:
                ax.set_ylabel(f'Predicted ({NS.UNIT})')
            k += 1
    fig.suptitle({'random': 'Random 10-fold CV (10 repeats)',
                  'spatial': 'Spatial block CV (hexagons, 4 grid positions)'}[cvs],
                 x=0.06, y=0.985, ha='left', fontsize=7.5, fontweight='bold')
    if scope == 'all':
        fig.legend(handles=[
            Line2D([], [], marker='o', ls='', color=NS.NPG[3], ms=2.5, label='Measured profile'),
            Line2D([], [], marker='o', ls='', color=NS.GREY, ms=2.5, label='Harmonised (Jobbágy)')],
            loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.01))
    NS.save(fig, os.path.join(C.FIG_DIR, fname))
    plt.show()


for cvs in C.CV_SCHEMES:
    tag = {'random': 'randomCV', 'spatial': 'spatialCV'}[cvs]
    make_figure('measured', cvs, f'Fig2_obs_vs_pred_measured_{tag}')
    make_figure('all', cvs, f'FigS2_obs_vs_pred_all_points_{tag}')

# ═══════════════════════════════════════════════════════════════════════════════
#  Random vs spatial CV — table + Fig. 2c (fold-level mean ± SD, measured profiles)
# ═══════════════════════════════════════════════════════════════════════════════
comp = []
for d in NS.DEPTH_ORDER:
    for m in MODELS:
        row = {'Depth': d, 'Model': m}
        for cvs in C.CV_SCHEMES:
            p = os.path.join(M.results_dir(d, cvs), f'summary_{m}.json')
            if os.path.exists(p):
                s = json.load(open(p))
                for k in ('r2', 'rmse', 'ccc'):
                    row[f'{cvs}_{k}'] = s[f'meas_{k}_mean']
                    row[f'{cvs}_{k}_sd'] = s[f'meas_{k}_sd']
                row[f'{cvs}_pooled_r2'] = s['pooled_measured']['r2']
                row[f'{cvs}_median_dist_km'] = s.get('median_test_train_km', np.nan)
        comp.append(row)
comp = pd.DataFrame(comp)
if {'random_r2', 'spatial_r2'} <= set(comp.columns):
    comp['delta_R2'] = comp['spatial_r2'] - comp['random_r2']
    comp['delta_RMSE_pct'] = 100 * (comp['spatial_rmse'] / comp['random_rmse'] - 1)
comp.to_csv(os.path.join(C.RESULTS_DIR, 'random_vs_spatialCV.csv'), index=False)
print(comp.round(3).to_string(index=False))

if {'random_r2', 'spatial_r2'} <= set(comp.columns):
    CVCOL = {'random': NS.GREY, 'spatial': NS.NPG[1]}
    nm = len(MODELS)
    fig, axs = plt.subplots(1, 3, figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.30),
                            gridspec_kw=dict(wspace=0.45))
    for ax, (k, lab) in zip(axs, [('r2', '$R^2$'), ('rmse', f'RMSE ({NS.UNIT})'),
                                  ('ccc', 'CCC')]):
        xt, xl = [], []
        for i, d in enumerate(NS.DEPTH_ORDER):
            for j, m in enumerate(MODELS):
                r = comp[(comp.Depth == d) & (comp.Model == m)].iloc[0]
                x0 = i * (nm + 1) + j
                for o, cvs in zip((-0.17, 0.17), C.CV_SCHEMES):
                    ax.errorbar(x0 + o, r[f'{cvs}_{k}'], yerr=r[f'{cvs}_{k}_sd'],
                                fmt=NS.MODEL_MARKER[m], ms=3.2,
                                color=CVCOL[cvs], mec='white', mew=0.3,
                                elinewidth=0.6, capsize=1.2)
                ax.plot([x0 - 0.17, x0 + 0.17], [r[f'random_{k}'], r[f'spatial_{k}']],
                        color=NS.LIGHT, lw=0.6, zorder=0)
            xt.append(i * (nm + 1) + (nm - 1) / 2)
            xl.append(NS.DEPTH_LABEL[d])
        ax.set_xticks(xt)
        ax.set_xticklabels(xl)
        ax.set_ylabel(lab)
        ax.tick_params(axis='x', length=0)
        NS.panel_letter(ax, 'abc'[list(axs).index(ax)], x=-0.30, y=1.02)
    from matplotlib.lines import Line2D as L2
    fig.legend(handles=[L2([], [], marker='o', ls='', color=NS.GREY, ms=3, label='Random CV'),
                        L2([], [], marker='o', ls='', color=NS.NPG[1], ms=3, label='Spatial block CV')]
                       + [L2([], [], marker=NS.MODEL_MARKER[m], ls='', color=NS.INK, ms=3,
                             label=NS.MODEL_LABEL[m]) for m in MODELS],
               loc='upper center', ncol=2 + len(MODELS), bbox_to_anchor=(0.5, 1.13))
    NS.save(fig, os.path.join(C.FIG_DIR, 'Fig2c_random_vs_spatialCV'))
    plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. S7 — QRF prediction-interval coverage (calibration of the uncertainty)
#  Empirical coverage (PICP) of the 50/80/90/95 % central intervals vs nominal,
#  held-out measured profiles, random and spatial CV. On the 1:1 line = well
#  calibrated; below = intervals too narrow; above = too wide.
# ═══════════════════════════════════════════════════════════════════════════════
if 'QRF' in MODELS:
    rows = []
    for cvs in C.CV_SCHEMES:
        for d in NS.DEPTH_ORDER:
            f = os.path.join(M.results_dir(d, cvs), 'qrf_interval_coverage.csv')
            if os.path.exists(f):
                rows.append(pd.read_csv(f).assign(CV=cvs, Depth=d))
    if rows:
        cov = pd.concat(rows, ignore_index=True)
        cov.to_csv(os.path.join(C.RESULTS_DIR, 'Table_QRF_interval_coverage.csv'), index=False)
        print(cov[cov.scope == 'measured'].round(2).to_string(index=False))
        CVCOL = {'random': NS.GREY, 'spatial': NS.NPG[1]}
        fig, axs = plt.subplots(1, 3, figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.33),
                                gridspec_kw=dict(wspace=0.35))
        for c, d in enumerate(NS.DEPTH_ORDER):
            ax = axs[c]
            ax.plot([40, 100], [40, 100], color=NS.INK, lw=0.6, ls='--')
            for cvs in C.CV_SCHEMES:
                s = cov[(cov.Depth == d) & (cov.CV == cvs) & (cov.scope == 'measured')]
                if len(s):
                    ax.plot(s.nominal, s.picp, '-o', color=CVCOL[cvs], ms=3, lw=0.9,
                            label={'random': 'Random CV', 'spatial': 'Spatial block CV'}[cvs])
                    w90 = s.loc[s.nominal == 90, 'mpiw'].values
                    if len(w90):
                        ax.text(0.97, 0.05 + 0.09 * C.CV_SCHEMES.index(cvs),
                                f'MPIW$_{{90}}$ = {w90[0]:.0f}', transform=ax.transAxes,
                                ha='right', fontsize=5.8, color=CVCOL[cvs])
            ax.set_xlim(45, 100)
            ax.set_ylim(40, 100)
            ax.set_xlabel('Nominal coverage (%)')
            if c == 0:
                ax.set_ylabel('Empirical coverage, PICP (%)')
                ax.legend(loc='upper left', fontsize=5.8)
            ax.set_title(f'QRF · {NS.DEPTH_LABEL[d]}', pad=3)
            NS.panel_letter(ax, 'abc'[c], x=-0.25, y=1.02)
        NS.save(fig, os.path.join(C.FIG_DIR, 'FigS7_QRF_interval_coverage'))
        plt.show()


## Step 3b — Maps of the spatial folds used + test→train distances

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 3b — MAPS OF THE SPATIAL BLOCK FOLDS ACTUALLY USED (from soc_spatial_cv)
#  Fig. S5 : first grid position per target + test→train distance curves
#  Fig. S6 : all grid positions used as repeats
# ═══════════════════════════════════════════════════════════════════════════════
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Polygon
from pyproj import Transformer
from scipy.spatial import cKDTree
from shapely.geometry import shape, Point, Polygon as ShPoly
from shapely.ops import unary_union

import nature_style as NS
import soc_config as C
import soc_ml as M
from soc_spatial import ORIGIN, block_keys, grid_shifts, test_train_dist

FONT = NS.setup_arial(verbose=False)
plt.rcParams.update({'axes.spines.top': True, 'axes.spines.right': True})
GEOJSON, FIG_DPI, WORK_DIR = C.GEOJSON, 600, C.FIG_DIR
os.makedirs(WORK_DIR, exist_ok=True)
TARGETS = [('SOC 0–30 cm', 'SOC30'), ('SOC 30–100 cm', 'DeepSOC'), ('SOC 0–100 cm', 'SOC100')]
TARGETS = [(lab, None, key, C.SPATIAL_DESIGN[key]) for lab, key in TARGETS]

FOLD_COLS = ["#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F",
             "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F", "#8C8C8C"]
LINE_COLS = {"map": "#000000", "cv": "#4E79A7", "blk": "#1B9E77"}

# land polygon in km and a 5 km grid of map cells
to_utm = Transformer.from_crs(4326, 32738, always_xy=True)
land = unary_union([shape(f["geometry"]) for f in json.load(open(GEOJSON))["features"]])
try:
    import shapely
    land_km = shapely.transform(land, lambda c: np.column_stack(to_utm.transform(c[:, 0], c[:, 1])) / 1000)
    inside_fn = lambda x, y: shapely.contains_xy(land_km, x, y)
except AttributeError:
    from shapely.ops import transform as sh_transform
    from shapely.prepared import prep
    land_km = sh_transform(lambda x, y, z=None: tuple(np.asarray(v) / 1000 for v in to_utm.transform(x, y)), land)
    inside_fn = lambda x, y: np.array([prep(land_km).contains(Point(a, b)) for a, b in zip(x, y)])

minx, miny, maxx, maxy = land_km.bounds
gx, gy = np.meshgrid(np.arange(np.floor(minx / 5) * 5 + 2.5, maxx, 5),
                     np.arange(np.floor(miny / 5) * 5 + 2.5, maxy, 5))
gx, gy = gx.ravel(), gy.ravel()
map_cells = np.c_[gx, gy][inside_fn(gx, gy)]


def block_polys(xy, fold, shape_, size, shift):
    keys = block_keys(xy, shape_, size, shift)
    o = ORIGIN + np.asarray(shift)
    out = []
    for a, b in np.unique(keys, axis=0):
        f = int(np.bincount(fold[(keys[:, 0] == a) & (keys[:, 1] == b)]).argmax())
        if shape_ == "square":
            x0, y0 = o + np.array([a, b]) * size
            poly = ShPoly([(x0, y0), (x0 + size, y0), (x0 + size, y0 + size), (x0, y0 + size)])
        else:
            r = size / np.sqrt(3)
            cx, cy = o[0] + r * np.sqrt(3) * (a + b / 2), o[1] + r * 1.5 * b
            ang = np.radians(30 + 60 * np.arange(6))
            poly = ShPoly(np.c_[cx + r * np.cos(ang), cy + r * np.sin(ang)])
        g = poly.intersection(land_km)
        if not g.is_empty:
            out.append((g, f))
    return out, len(np.unique(keys, axis=0))


def patches(g, **kw):
    parts = g.geoms if hasattr(g, "geoms") else [g]
    return [Polygon(np.asarray(p.exterior.coords), closed=True, **kw) for p in parts if isinstance(p, ShPoly)]


def draw_map(ax, P, rep, point_size):
    ax.set_xlim(minx - 8, maxx + 8)
    ax.set_ylim(miny - 8, maxy + 8)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_linewidth(0.6)
        sp.set_color("#333333")
    for p in patches(land_km, fc="#EFEFEF", ec="#8c8c8c", lw=0.25):
        ax.add_patch(p)
    for g, f in rep["polys"]:
        for p in patches(g, fc=FOLD_COLS[f - 1], ec="#404040", lw=0.3, alpha=0.30):
            ax.add_patch(p)
    order = np.random.default_rng(1).permutation(len(P))
    ax.scatter(P[order, 0], P[order, 1], c=[FOLD_COLS[f - 1] for f in rep["fold"][order]],
               s=point_size, linewidths=0.12, edgecolors="#262626", zorder=3)
    ax.plot([maxx - 196, maxx + 4], [miny + 22] * 2, "k-", lw=1.6, solid_capstyle="butt", zorder=5)
    ax.text(maxx - 96, miny + 30, "200 km", ha="center", va="bottom", fontsize=6.5, zorder=5)


def fold_legend(fig, height):
    ax = fig.add_axes([0, 0, 1, height])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(0.5, 0.72, "Test fold (spatial blocks)", ha="center", va="center", fontsize=10, fontweight="bold")
    xs = 0.5 + (np.arange(10) - 4.5) * 0.072
    ax.scatter(xs, [0.32] * 10, c=FOLD_COLS, s=50, edgecolor="#262626", linewidth=0.4)
    for i, x in enumerate(xs):
        ax.text(x + 0.013, 0.32, str(i + 1), va="center", fontsize=9)


# read back the folds that were used in the models
fig_data = []
for label, _, response, (shape_, size) in TARGETS:
    path = os.path.join(M.results_dir(response, "spatial"), "spatial_folds.csv")
    if not os.path.exists(path):
        continue
    t = pd.read_csv(path)
    P = t[["x_km", "y_km"]].values
    rep_cols = sorted([c for c in t.columns if c.startswith("fold_rep")], key=lambda c: int(c[8:]))
    random_folds = np.random.default_rng(1).permutation(np.resize(np.arange(1, 11), len(P)))
    d = dict(label=label, shape=shape_, size=size, P=P,
             map_dist=cKDTree(P).query(map_cells)[0], random_dist=test_train_dist(P, random_folds), reps=[])
    for i, c in enumerate(rep_cols):
        fold = t[c].values.astype(int)
        shift = grid_shifts(size)[i]
        polys, n_blocks = block_polys(P, fold, shape_, size, shift)
        d["reps"].append(dict(fold=fold, polys=polys, n_blocks=n_blocks, shift=shift, dist=test_train_dist(P, fold)))
    fig_data.append(d)

letters = "abcdefghijkl"

# figure 1: first grid position for each target, with the distance curves
nc = len(fig_data)
fig = plt.figure(figsize=(7.2, 8.6))
w = 0.96 / nc - 0.02
for j, d in enumerate(fig_data):
    rep, x0 = d["reps"][0], 0.02 + j * 0.96 / nc
    fig.text(x0 + w / 2, 0.975, d["label"], ha="center", va="center", fontsize=10.5, fontweight="bold")
    fig.text(x0 + w / 2, 0.952, f"n = {len(d['P']):,} plots", ha="center", va="center", fontsize=7.5, color="#4d4d4d")
    fig.text(x0, 0.915, letters[j], ha="left", va="center", fontsize=19, fontweight="bold")
    fig.text(x0 + 0.06, 0.925, f"Median dist.: {np.median(rep['dist']):.1f} km", ha="left", va="center", fontsize=7)
    fig.text(x0 + 0.06, 0.905, f"{d['shape']}s {d['size']:g} km | {rep['n_blocks']} blocks",
             ha="left", va="center", fontsize=6.6, color="#4d4d4d")
    draw_map(fig.add_axes([x0, 0.37, w, 0.52]), d["P"], rep, 2.0 if len(d["P"]) > 3000 else 3.0)

    fig.text(x0, 0.315, letters[nc + j], ha="left", va="center", fontsize=19, fontweight="bold")
    fig.text(x0 + 0.06, 0.315, f"Map → plot: {np.median(d['map_dist']):.0f} km", ha="left", va="center", fontsize=7)
    ax = fig.add_axes([x0 + 0.055, 0.115, w - 0.07, 0.175])
    ax.set_xscale("log")
    ax.set_xlim(0.01, 600)
    ax.set_ylim(0, 1)
    for key, dist, lw, lab in [("map", d["map_dist"], 2.0, "Map cells"),
                               ("cv", d["random_dist"], 1.2, "Random CV"),
                               ("blk", rep["dist"], 1.6, "Spatial blocks")]:
        v = np.sort(np.maximum(dist, 0.01))
        ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=LINE_COLS[key], lw=lw, label=lab, drawstyle="steps-post")
    ax.set_xticks([0.01, 0.1, 1, 10, 100])
    ax.set_xticklabels(["0.01", "0.1", "1", "10", "100"], fontsize=7)
    ax.tick_params(axis="y", labelsize=7, length=2.5)
    ax.tick_params(axis="x", length=2.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("km (log scale)", fontsize=7.5, labelpad=2)
    if j == 0:
        ax.set_ylabel("Cumulative share", fontsize=7.5, labelpad=2)
    ax.legend(loc="lower right", fontsize=6, frameon=True, framealpha=0.9, edgecolor="none",
              handlelength=1.3, labelspacing=0.15, borderaxespad=0.1, borderpad=0.25)
fig.text(0.5, 0.345, "Distance from test plots to the nearest training plot",
         ha="center", va="center", fontsize=9, fontweight="bold")
fold_legend(fig, 0.065)
NS.save(fig, os.path.join(WORK_DIR, "FigS5_spatialCV_blocks_used"))
plt.show()

# figure 2: all grid positions used as repeats
ncol = max(len(d["reps"]) for d in fig_data)
if ncol > 1:
    wm, hm = 1.55, 1.55 * 1.95
    W, H = 0.45 + ncol * (wm + 0.08), 0.35 + len(fig_data) * (hm + 0.45) + 0.7
    fig = plt.figure(figsize=(W, H))
    k = 0
    for r, d in enumerate(fig_data):
        yb = H - 0.35 - (r + 1) * (hm + 0.45)
        fig.text(0.2 / W, (yb + hm / 2) / H, d["label"], rotation=90, ha="center", va="center",
                 fontsize=10.5, fontweight="bold")
        for c, rep in enumerate(d["reps"]):
            xb = 0.45 + c * (wm + 0.08)
            if r == 0:
                fig.text((xb + wm / 2) / W, (H - 0.18) / H, f"Repeat {c + 1}", ha="center", va="center",
                         fontsize=10.5, fontweight="bold")
            fig.text(xb / W, (yb + hm + 0.22) / H, letters[k] if k < len(letters) else "",
                     ha="left", va="center", fontsize=15, fontweight="bold")
            fig.text((xb + 0.28) / W, (yb + hm + 0.28) / H,
                     f"shift {rep['shift'][0]:g}, {rep['shift'][1]:g} km | {rep['n_blocks']} blocks",
                     ha="left", va="center", fontsize=6.3)
            fig.text((xb + 0.28) / W, (yb + hm + 0.12) / H, f"Median dist.: {np.median(rep['dist']):.1f} km",
                     ha="left", va="center", fontsize=6.3, color="#4d4d4d")
            draw_map(fig.add_axes([xb / W, yb / H, wm / W, hm / H]), d["P"], rep,
                     1.2 if len(d["P"]) > 3000 else 1.8)
            k += 1
    fold_legend(fig, 0.6 / H)
    NS.save(fig, os.path.join(WORK_DIR, "FigS6_spatialCV_all_grid_positions"))
    plt.show()

## Step 4 — SHAP importance, beeswarm, SHAP dependence + PDP, categorical SHAP

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — SHAP (out-of-fold) + PARTIAL DEPENDENCE, Nature-Reviews style
#   XGBoost : GPU TreeSHAP  (booster.predict(..., pred_contribs=True), device=cuda)
#   RF      : sklearn twin with the same tuned hyper-parameters + shap.TreeExplainer
#  Fig. 3  : mean |SHAP| importance, RF vs XGBoost, 3 depths
#  Fig. 4  : SHAP beeswarm, 2 × 3
#  Fig. 5/6: SHAP dependence (LOWESS ± 95% CI) with centred PDP overlay — per model
#  Fig. S3/S4 : SHAP by land-cover / soil class
# ═══════════════════════════════════════════════════════════════════════════════
import os
import textwrap
import warnings

import joblib
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
from statsmodels.nonparametric.smoothers_lowess import lowess

import nature_style as NS
import soc_config as C
import soc_ml as M

warnings.filterwarnings('ignore')
FONT = NS.setup_arial(verbose=False)

MODELS = C.MODELS                  # QRF is explained through its conditional mean
N_FOLDS, N_SEEDS = (3, 1) if C.FAST else (10, 3)   # OOF SHAP: 3 × 10-fold
TOP_K_DEP = 4                     # numeric features shown in dependence figure
PDP_GRID, PDP_SAMPLE = (15, 100) if C.FAST else (40, 600)
SHAP_DIR = os.path.join(C.RESULTS_DIR, 'SHAP')
os.makedirs(SHAP_DIR, exist_ok=True)
USE_GPU = M.gpu_available()
print(f'GPU TreeSHAP for XGBoost: {USE_GPU} | final models from {C.FINAL_MODEL_CV} CV')

NUM = C.NUMERIC_PREDICTORS
CAT_ENC = [f'{c}_enc' for c in C.CATEGORICAL_PREDICTORS]


# ── SHAP for one fold ─────────────────────────────────────────────────────────
def fold_shap(model_name, params, X, y, tr, te, seed):
    if model_name == 'XGBoost':
        mdl = M.make_model('XGBoost', params, USE_GPU, seed=seed)
        mdl.fit(X[tr], y[tr])
        bst = mdl.get_booster()
        contrib = bst.predict(xgb.DMatrix(X[te]), pred_contribs=True)
        return contrib[:, :-1], contrib[:, -1]
    import shap
    mdl = M.to_sklearn_rf(params, seed=seed)
    mdl.set_params(n_jobs=1)
    mdl.fit(X[tr], y[tr])
    ex = shap.TreeExplainer(mdl, feature_perturbation='tree_path_dependent')
    sv = ex.shap_values(X[te], check_additivity=False)
    return np.asarray(sv), np.full(len(te), float(np.ravel(ex.expected_value)[0]))


def oof_shap(model_name, params, X, y):
    tasks = []
    for s in range(N_SEEDS):
        for tr, te in KFold(N_FOLDS, shuffle=True, random_state=137 * s + 1).split(X):
            tasks.append((s, tr, te))
    n_jobs = 1 if (model_name == 'XGBoost' and USE_GPU) else -1
    res = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(fold_shap)(model_name, params, X, y, tr, te, 1000 * s + i)
        for i, (s, tr, te) in enumerate(tasks))
    sv = np.vstack([r[0] for r in res])
    base = np.concatenate([r[1] for r in res])
    idx = np.concatenate([t[2] for t in tasks])
    return sv, base, idx


def pdp_curves(model, X, feat_idx, rng):
    """Centred partial dependence (mean ICE − mean prediction) + 10–90 % ICE band."""
    sub = X[rng.choice(len(X), min(PDP_SAMPLE, len(X)), replace=False)]
    base = M.predict(model, sub).mean()
    out = {}
    for j in feat_idx:
        grid = np.linspace(*np.percentile(X[:, j], [2, 98]), PDP_GRID)
        ice = np.empty((len(grid), len(sub)))
        Xg = np.repeat(sub[None], len(grid), 0)
        Xg[:, :, j] = grid[:, None]
        ice[:] = M.predict(model, Xg.reshape(-1, X.shape[1])).reshape(len(grid), -1)
        ice_c = ice - ice.mean(0, keepdims=True) + (ice.mean() - base)
        out[j] = dict(grid=grid, pdp=ice.mean(1) - base,
                      lo=np.percentile(ice_c, 10, 1), hi=np.percentile(ice_c, 90, 1))
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  Compute
# ═══════════════════════════════════════════════════════════════════════════════
df, features, encoders = M.load_harmonized_table()
X = df[features].values.astype(np.float32)
R = {}
for d in NS.DEPTH_ORDER:
    y, _ = M.full_target(df, d)
    for m in MODELS:
        pkl = os.path.join(M.results_dir(d, C.FINAL_MODEL_CV), 'models', f'{m}.pkl')
        params = joblib.load(pkl)['best_params']
        print(f'SHAP  {m:<12} | {d:<7} | params {params}', flush=True)
        sv, base, idx = oof_shap(m, params, X, y)
        full = (M.to_sklearn_rf(params) if m in ('RandomForest', 'QRF')
                else M.make_model(m, params, USE_GPU)).fit(X, y)
        rng = np.random.default_rng(0)
        imp = np.abs(sv).mean(0)
        top_num = [features.index(f) for f in
                   sorted(NUM, key=lambda f: -imp[features.index(f)])[:TOP_K_DEP]]
        pdp = pdp_curves(full, X, top_num, rng)
        R[(m, d)] = dict(sv=sv, Xv=X[idx], idx=idx, imp=imp,
                         imp_sd=np.array([np.abs(sv[s * len(X):(s + 1) * len(X)]).mean(0)
                                         for s in range(N_SEEDS)]).std(0),
                         top_num=top_num, pdp=pdp)
        pd.DataFrame({'feature': features, 'label': [NS.flabel(f) for f in features],
                      'mean_abs_shap': imp,
                      'share_pct': 100 * imp / imp.sum()}).sort_values(
            'mean_abs_shap', ascending=False).to_csv(
            os.path.join(SHAP_DIR, f'shap_importance_{m}_{d}.csv'), index=False)
        np.savez_compressed(os.path.join(SHAP_DIR, f'shap_values_{m}_{d}.npz'),
                            shap=sv, X=X[idx], row_id=idx, base=base, features=features)

# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. 3 — importance (grouped horizontal bars, RF vs XGBoost)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axs = plt.subplots(1, 3, figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.36),
                        gridspec_kw=dict(wspace=0.75))
for c, d in enumerate(NS.DEPTH_ORDER):
    ax = axs[c]
    rel = {m: 100 * R[(m, d)]['imp'] / R[(m, d)]['imp'].sum() for m in MODELS}
    rel_sd = {m: 100 * R[(m, d)]['imp_sd'] / R[(m, d)]['imp'].sum() for m in MODELS}
    order = np.argsort(-sum(rel[m] for m in MODELS))
    ypos = np.arange(len(features))[::-1]
    hb = 0.8 / len(MODELS)
    for k, m in enumerate(MODELS):
        ax.barh(ypos + (len(MODELS) - 1) / 2 * hb - k * hb, rel[m][order], height=hb * 0.95,
                xerr=rel_sd[m][order], color=NS.MODEL_COLOR[m], label=NS.MODEL_LABEL[m],
                error_kw=dict(lw=0.5, capsize=1, capthick=0.5, ecolor=NS.INK))
    ax.set_yticks(ypos)
    ax.set_yticklabels([NS.flabel(features[i]) for i in order])
    ax.set_xlabel('Relative importance,\nmean |SHAP| (%)')
    ax.set_title(NS.DEPTH_LABEL[d], pad=3)
    ax.tick_params(axis='y', length=0)
    NS.panel_letter(ax, 'abc'[c], x=-0.55, y=1.02)
axs[0].legend(loc='lower right', fontsize=5.8)
NS.save(fig, os.path.join(C.FIG_DIR, 'Fig3_SHAP_importance'))
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. 4 — beeswarm (2 × 3)
# ═══════════════════════════════════════════════════════════════════════════════
BEE = mcolors.LinearSegmentedColormap.from_list('bee', ['#3C5488', '#B7C3DE', '#F2C6BE', '#E64B35'])


def beeswarm(ax, sv, Xv, max_pts=2500, rng=np.random.default_rng(3)):
    imp = np.abs(sv).mean(0)
    order = np.argsort(imp)                      # least important at bottom
    for row, j in enumerate(order):
        s = sv[:, j]
        sel = rng.choice(len(s), min(max_pts, len(s)), replace=False)
        s, xv = s[sel], Xv[sel, j]
        # jitter proportional to local density (histogram-based)
        bins = np.linspace(s.min(), s.max() + 1e-9, 60)
        b = np.digitize(s, bins)
        cnt = np.bincount(b, minlength=len(bins) + 1)[b]
        jit = rng.uniform(-1, 1, len(s)) * 0.38 * cnt / cnt.max()
        if features[j] in CAT_ENC:
            col = NS.GREY
        else:
            lo, hi = np.percentile(xv, [5, 95])
            col = BEE(np.clip((xv - lo) / (hi - lo + 1e-12), 0, 1))
        ax.scatter(s, row + jit, c=col, s=0.8, lw=0, rasterized=True)
    ax.axvline(0, color=NS.INK, lw=0.5)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([NS.flabel(features[j]) for j in order])
    ax.tick_params(axis='y', length=0)
    ax.spines['left'].set_visible(False)


fig, axs = plt.subplots(len(MODELS), 3, squeeze=False,
                        figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.31 * len(MODELS)),
                        gridspec_kw=dict(wspace=0.65, hspace=0.42))
k = 0
for r, m in enumerate(MODELS):
    for c, d in enumerate(NS.DEPTH_ORDER):
        ax = axs[r, c]
        beeswarm(ax, R[(m, d)]['sv'], R[(m, d)]['Xv'])
        ax.set_title(f'{NS.MODEL_LABEL[m]} · {NS.DEPTH_LABEL[d]}', pad=3)
        if r == len(MODELS) - 1:
            ax.set_xlabel(f'SHAP value ({NS.UNIT})')
        NS.panel_letter(ax, 'abcdefghi'[k], x=-0.52, y=1.02)
        k += 1
cax = fig.add_axes([0.93, 0.35, 0.008, 0.3])
cb = fig.colorbar(plt.cm.ScalarMappable(cmap=BEE), cax=cax, ticks=[0, 1])
cb.ax.set_yticklabels(['Low', 'High'])
cb.set_label('Feature value', labelpad=-4)
cb.outline.set_linewidth(0.4)
NS.save(fig, os.path.join(C.FIG_DIR, 'Fig4_SHAP_beeswarm'))
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. 5/6 — SHAP dependence + centred PDP (rows = depth, cols = top features)
# ═══════════════════════════════════════════════════════════════════════════════
def lowess_ci(x, y, frac=0.3, n_boot=100, rng=np.random.default_rng(7)):
    grid = np.linspace(*np.percentile(x, [2, 98]), 60)
    fit = lambda xx, yy: np.interp(grid, *lowess(yy, xx, frac=frac, return_sorted=True).T)
    main = fit(x, y)
    sub = min(len(x), 2000)
    boots = []
    for _ in range(n_boot):
        i = rng.choice(len(x), sub, replace=True)
        boots.append(fit(x[i], y[i]))
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    return grid, main, lo, hi


def dependence_figure(m):
    fig = plt.figure(figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.78))
    outer = gridspec.GridSpec(3, TOP_K_DEP, figure=fig, wspace=0.42, hspace=0.62)
    k = 0
    for r, d in enumerate(NS.DEPTH_ORDER):
        res = R[(m, d)]
        for c, j in enumerate(res['top_num']):
            inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[r, c],
                                                     height_ratios=[1, 4.5], hspace=0.05)
            axh, ax = fig.add_subplot(inner[0]), fig.add_subplot(inner[1])
            x, s = res['Xv'][:, j], res['sv'][:, j]
            lo_x, hi_x = np.percentile(x, [1, 99])
            keep = (x >= lo_x) & (x <= hi_x)
            x, s = x[keep], s[keep]
            ax.scatter(x, s, s=1.2, color=NS.MODEL_COLOR[m], alpha=0.18, lw=0,
                       rasterized=True, zorder=1)
            g, mu, lo, hi = lowess_ci(x, s)
            ax.fill_between(g, lo, hi, color=NS.INK, alpha=0.18, lw=0, zorder=2)
            ax.plot(g, mu, color=NS.INK, lw=1.0, zorder=3, label='SHAP (LOWESS ± 95% CI)')
            p = res['pdp'][j]
            # align PDP with SHAP: zero mean over the observed distribution of x
            shift = np.mean(np.interp(x, p['grid'], p['pdp']))
            p = dict(grid=p['grid'], pdp=p['pdp'] - shift, lo=p['lo'] - shift,
                     hi=p['hi'] - shift)
            ax.fill_between(p['grid'], p['lo'], p['hi'], color=NS.NPG[2], alpha=0.12,
                            lw=0, zorder=1)
            ax.plot(p['grid'], p['pdp'], color=NS.NPG[2], lw=1.0, ls='--', zorder=4,
                    label='Centred PDP (ICE 10–90%)')
            ax.axhline(0, color=NS.GREY, lw=0.4, ls=':', zorder=0)
            ylo, yhi = np.percentile(s, [0.5, 99.5])
            pad = 0.1 * (yhi - ylo)
            ax.set_ylim(ylo - pad, yhi + pad)
            ax.set_xlim(lo_x, hi_x)
            ax.set_xlabel(NS.flabel(features[j]), labelpad=1.5)
            ax.locator_params(nbins=4)
            if c == 0:
                ax.set_ylabel(f'{NS.DEPTH_LABEL[d]}\nSHAP ({NS.UNIT})')
            axh.hist(x, bins=35, color=NS.LIGHT, edgecolor='none')
            axh.set_xlim(lo_x, hi_x)
            axh.axis('off')
            NS.panel_letter(axh, 'abcdefghijkl'[k], x=-0.30 if c else -0.42, y=0.4)
            k += 1
            if r == 0 and c == TOP_K_DEP - 1:
                h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.95))
    fig.suptitle(NS.MODEL_LABEL[m], x=0.125, y=0.975, ha='left', fontsize=7.5,
                 fontweight='bold')
    NS.save(fig, os.path.join(C.FIG_DIR, f'Fig5{"abc"[MODELS.index(m)]}_SHAP_PDP_dependence_{m}'))
    plt.show()


for m in MODELS:
    dependence_figure(m)


# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. S3/S4 — SHAP by category (land cover, soil type)
# ═══════════════════════════════════════════════════════════════════════════════
def categorical_figure(m):
    fig, axs = plt.subplots(len(CAT_ENC), 3,
                            figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.75),
                            gridspec_kw=dict(wspace=0.12, hspace=0.35))
    k = 0
    for r, (cenc, craw) in enumerate(zip(CAT_ENC, C.CATEGORICAL_PREDICTORS)):
        classes = encoders[craw].classes_
        j = features.index(cenc)
        # common class order (median SHAP for SOC100)
        ref = R[(m, 'SOC100')]
        med = [np.median(ref['sv'][ref['Xv'][:, j] == i, j]) if (ref['Xv'][:, j] == i).any()
               else 0 for i in range(len(classes))]
        order = np.argsort(med)
        for c, d in enumerate(NS.DEPTH_ORDER):
            ax = axs[r, c]
            res = R[(m, d)]
            data = [res['sv'][res['Xv'][:, j] == i, j] for i in order]
            bp = ax.boxplot(data, vert=False, widths=0.6, showfliers=False,
                            patch_artist=True, medianprops=dict(color=NS.INK, lw=0.8),
                            whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5),
                            boxprops=dict(lw=0.5))
            for patch, dd in zip(bp['boxes'], data):
                med_v = np.median(dd) if len(dd) else 0
                patch.set_facecolor(NS.NPG[0] if med_v > 0 else NS.NPG[3])
                patch.set_alpha(0.55)
            ax.axvline(0, color=NS.GREY, lw=0.5, ls=':')
            n_per = [len(dd) for dd in data]
            if c == 0:
                ax.set_yticklabels([f'{textwrap.shorten(classes[i], 30)} ({n})'
                                    for i, n in zip(order, n_per)], fontsize=5.6)
            else:
                ax.set_yticklabels([])
            ax.tick_params(axis='y', length=0)
            ax.set_title(f'{NS.flabel(craw)} · {NS.DEPTH_LABEL[d]}', pad=3)
            if r == len(CAT_ENC) - 1:
                ax.set_xlabel(f'SHAP value ({NS.UNIT})')
            NS.panel_letter(ax, 'abcdef'[k], x=-0.08 if c else -0.75, y=1.02)
            k += 1
    fig.suptitle(NS.MODEL_LABEL[m], x=0.02, y=0.99, ha='left', fontsize=7.5,
                 fontweight='bold')
    NS.save(fig, os.path.join(C.FIG_DIR, f'FigS3{"abc"[MODELS.index(m)]}_SHAP_categorical_{m}'))
    plt.show()


for m in MODELS:
    categorical_figure(m)
print('✅ SHAP done →', SHAP_DIR)


## Step 5 — GPU bootstrap maps (RF, XGBoost): mean / SD / CV

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 5 — GPU MAPPING (launch) + MAP FIGURES
# ═══════════════════════════════════════════════════════════════════════════════
import json
import os
import subprocess
import sys
import time

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling

import nature_style as NS
import soc_config as C

RUN_MAPPING = True                       # False → only re-draw figures
TARGETS_TO_MAP = ['SOC30', 'DeepSOC', 'SOC100']
QUEUES = {0: 'RandomForest', 1: 'XGBoost'}
N_BOOT = C.N_BOOT_MAP

# ── 5a. launch: RF on GPU 0, XGBoost on GPU 1 ─────────────────────────────────
if RUN_MAPPING:
    try:
        import cupy as cp
        n_gpu = cp.cuda.runtime.getDeviceCount()
    except Exception:                                            # noqa: BLE001
        n_gpu = 0
    os.makedirs(C.MAP_DIR, exist_ok=True)
    SRC = os.path.dirname(os.path.abspath(C.__file__))
    procs = {}
    for gpu, model in QUEUES.items():
        cmd = ' ; '.join(f'{sys.executable} -u {SRC}/map_soc_gpu.py --target {t} '
                          f'--model {model} --n-boot {N_BOOT}' for t in TARGETS_TO_MAP)
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu if n_gpu > 1 else 0)
        env['PYTHONPATH'] = SRC + os.pathsep + env.get('PYTHONPATH', '')
        log = open(os.path.join(C.MAP_DIR, f'log_map_{model}.txt'), 'w')
        procs[model] = (subprocess.Popen(cmd, shell=True, env=env, stdout=log,
                                         stderr=subprocess.STDOUT, cwd=SRC), log)
        print(f'Launched mapping {model} on GPU {env["CUDA_VISIBLE_DEVICES"]}')
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(60)
        for model, (p, log) in procs.items():
            lines = [l.strip() for l in open(log.name) if l.strip()]
            print(f'[{model:<12}] {lines[-1] if lines else ""}', flush=True)
    for model, (p, log) in procs.items():
        log.close()
        print(f'{model}: exit {p.returncode}')

# ── 5b. stock totals + additivity check (SOC30 + SOC30–100 vs SOC0–100) ──────
print('\nNational totals (bootstrap-mean maps)')
for m in QUEUES.values():
    tot = {}
    for d in NS.DEPTH_ORDER:
        f = os.path.join(C.MAP_DIR, f'{d}_{m}', 'map_stats.json')
        if os.path.exists(f):
            tot[d] = json.load(open(f))['total_stock_PgC']
            print(f'  {m:<12} {d:<8} {tot[d]:7.3f} Pg C')
    if len(tot) == 3:
        diff = tot['SOC30'] + tot['DeepSOC'] - tot['SOC100']
        print(f'  {m:<12} SOC30+Deep − SOC100 = {diff:+.3f} Pg C '
              f'({100 * diff / tot["SOC100"]:+.1f} %)')

# ── 5c. Figure — 3 depths × (mean, SD, CV), one per model ────────────────────
FONT = NS.setup_arial(verbose=False)
DECIM = 8                                      # read every 8th pixel for plotting

CMAPS = {
    'mean': mcolors.LinearSegmentedColormap.from_list(
        'soc', ['#FFF7E3', '#F6D58A', '#D99A3E', '#9C5A1E', '#5A2F12', '#2B1708']),
    'sd': mcolors.LinearSegmentedColormap.from_list(
        'sd', ['#F7FBFF', '#C6DBEF', '#6BAED6', '#2171B5', '#08306B']),
    'cv': mcolors.LinearSegmentedColormap.from_list(
        'cv', ['#FCFBFD', '#DADAEB', '#9E9AC8', '#6A51A3', '#3F007D']),
}
STAT_LABEL = {'mean': f'Mean ({NS.UNIT})', 'sd': f'SD ({NS.UNIT})', 'cv': 'CV (%)'}


def read_small(path):
    with rasterio.open(path) as src:
        arr = src.read(1, out_shape=(src.height // DECIM, src.width // DECIM),
                       resampling=Resampling.nearest).astype(float)
        b = src.bounds
        nd = src.nodata
    arr[(arr == nd) | ~np.isfinite(arr)] = np.nan
    return arr, (b.left, b.right, b.bottom, b.top)


for m in QUEUES.values():
    fig, axs = plt.subplots(3, 3, figsize=(NS.DOUBLE_COL * 0.82, NS.DOUBLE_COL * 1.12),
                            gridspec_kw=dict(wspace=0.05, hspace=0.12))
    k = 0
    for r, d in enumerate(NS.DEPTH_ORDER):
        for c, s in enumerate(['mean', 'sd', 'cv']):
            ax = axs[r, c]
            p = os.path.join(C.MAP_DIR, f'{d}_{m}', f'{d}_{m}_{s}.tif')
            if not os.path.exists(p):
                ax.set_axis_off()
                continue
            arr, ext = read_small(p)
            vmax = np.nanpercentile(arr, 98)
            im = ax.imshow(arr, extent=ext, cmap=CMAPS[s], vmin=0, vmax=vmax,
                           interpolation='nearest')
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
            cax = ax.inset_axes([0.84, 0.06, 0.035, 0.36])
            cb = fig.colorbar(im, cax=cax, extend='max')
            cb.outline.set_linewidth(0.3)
            cb.ax.tick_params(labelsize=5.2, width=0.3, length=1.5)
            cb.set_label(STAT_LABEL[s], fontsize=5.4, labelpad=1)
            cb.ax.yaxis.set_label_position('left')
            if r == 0:
                ax.set_title({'mean': 'Mean', 'sd': 'Standard deviation',
                              'cv': 'Coefficient of variation'}[s], loc='center', pad=3)
            if c == 0:
                ax.text(-0.04, 0.5, NS.DEPTH_LABEL[d], transform=ax.transAxes,
                        rotation=90, va='center', ha='right', fontsize=7)
            NS.panel_letter(ax, 'abcdefghi'[k], x=0.0, y=0.93)
            k += 1
    fig.suptitle(f'{NS.MODEL_LABEL[m]} · bootstrap ensemble (B = {N_BOOT})',
                 x=0.13, y=0.915, ha='left', fontsize=7.5, fontweight='bold')
    NS.save(fig, os.path.join(C.FIG_DIR, f'Fig7_maps_{m}'))
    plt.show()


## Step 6 — QRF prediction with prediction-interval rasters (90 % PI)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 6 — QRF PREDICTION WITH PREDICTION-INTERVAL RASTERS (90 % PI)
#  For each depth: mean, q05, q50, q95, PI width (q95 − q05) and relative width
#  GeoTIFFs + Fig. 8 (lower bound, median, upper bound, PI width).
#  Runs on CPU (quantile-forest); the three depths run in parallel processes.
# ═══════════════════════════════════════════════════════════════════════════════
import json
import os
import subprocess
import sys
import time

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling

import nature_style as NS
import soc_config as C

RUN_QRF_MAPS = True                 # False → only re-draw the figure
TARGETS_TO_MAP = ['SOC30', 'DeepSOC', 'SOC100']
DECIM = C.QRF_MAP_DECIM             # e.g. 2 or 4 to predict on a coarser grid
N_PARALLEL = 1                      # processes at once (each uses all cores)

if RUN_QRF_MAPS:
    SRC = os.path.dirname(os.path.abspath(C.__file__))
    os.makedirs(C.MAP_DIR, exist_ok=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = SRC + os.pathsep + env.get('PYTHONPATH', '')
    queue, running = list(TARGETS_TO_MAP), {}
    while queue or running:
        while queue and len(running) < N_PARALLEL:
            t = queue.pop(0)
            log = open(os.path.join(C.MAP_DIR, f'log_qrf_{t}.txt'), 'w')
            p = subprocess.Popen([sys.executable, '-u', f'{SRC}/map_qrf_intervals.py',
                                  '--target', t, '--decim', str(DECIM)],
                                 env=env, stdout=log, stderr=subprocess.STDOUT, cwd=SRC)
            running[t] = (p, log)
            print(f'Launched QRF interval map for {t}')
        time.sleep(30)
        for t, (p, log) in list(running.items()):
            lines = [l.strip() for l in open(log.name) if l.strip()]
            print(f'[QRF {t:<8}] {lines[-1] if lines else ""}', flush=True)
            if p.poll() is not None:
                log.close()
                print(f'QRF {t}: exit {p.returncode}')
                running.pop(t)

# ── totals ────────────────────────────────────────────────────────────────────
print('\nQRF national totals (mean map; bounds = sums of pixel quantiles)')
for d in NS.DEPTH_ORDER:
    f = os.path.join(C.MAP_DIR, f'{d}_QRF', 'map_stats.json')
    if os.path.exists(f):
        s = json.load(open(f))
        print(f'  {d:<8} {s["total_stock_PgC"]:7.3f} Pg C   '
              f'[{s["total_stock_PgC_sum_of_lower"]:.3f}, {s["total_stock_PgC_sum_of_upper"]:.3f}]')

# ═══════════════════════════════════════════════════════════════════════════════
#  Fig. 8 — QRF: lower bound, median, upper bound, 90 % PI width
#  q05 / q50 / q95 share one colour scale per depth so they can be compared.
# ═══════════════════════════════════════════════════════════════════════════════
FONT = NS.setup_arial(verbose=False)
PLOT_DECIM = 8
lo_q, hi_q = C.QRF_PI
level = int(round(100 * (hi_q - lo_q)))
COLS = [(f'q{100 * lo_q:02.0f}', f'Lower bound (q{100 * lo_q:02.0f})'),
        ('q50', 'Median (q50)'),
        (f'q{100 * hi_q:02.0f}', f'Upper bound (q{100 * hi_q:02.0f})'),
        (f'piw{level}', f'{level}% PI width')]
SOC_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'soc', ['#FFF7E3', '#F6D58A', '#D99A3E', '#9C5A1E', '#5A2F12', '#2B1708'])
PIW_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'piw', ['#FCFBFD', '#DADAEB', '#9E9AC8', '#6A51A3', '#3F007D'])


def read_small(path):
    with rasterio.open(path) as src:
        arr = src.read(1, out_shape=(max(1, src.height // PLOT_DECIM), max(1, src.width // PLOT_DECIM)),
                       resampling=Resampling.nearest).astype(float)
        b, nd = src.bounds, src.nodata
    arr[(arr == nd) | ~np.isfinite(arr)] = np.nan
    return arr, (b.left, b.right, b.bottom, b.top)


fig, axs = plt.subplots(3, 4, figsize=(NS.DOUBLE_COL, NS.DOUBLE_COL * 0.86),
                        gridspec_kw=dict(wspace=0.04, hspace=0.12))
k = 0
for r, d in enumerate(NS.DEPTH_ORDER):
    paths = {key: os.path.join(C.MAP_DIR, f'{d}_QRF', f'{d}_QRF_{key}.tif') for key, _ in COLS}
    if not all(os.path.exists(p) for p in paths.values()):
        for ax in axs[r]:
            ax.set_axis_off()
        continue
    data = {key: read_small(p) for key, p in paths.items()}
    vmax_soc = np.nanpercentile(data[COLS[2][0]][0], 98)
    for c, (key, title) in enumerate(COLS):
        ax = axs[r, c]
        arr, ext = data[key]
        is_w = key.startswith('piw')
        im = ax.imshow(arr, extent=ext, cmap=PIW_CMAP if is_w else SOC_CMAP, vmin=0,
                       vmax=np.nanpercentile(arr, 98) if is_w else vmax_soc,
                       interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c in (2, 3):              # one bar for the shared SOC scale, one for width
            cax = ax.inset_axes([0.84, 0.06, 0.035, 0.36])
            cb = fig.colorbar(im, cax=cax, extend='max')
            cb.outline.set_linewidth(0.3)
            cb.ax.tick_params(labelsize=5.2, width=0.3, length=1.5)
            cb.set_label(NS.UNIT, fontsize=5.4, labelpad=1)
            cb.ax.yaxis.set_label_position('left')
        if r == 0:
            ax.set_title(title, loc='center', pad=3)
        if c == 0:
            ax.text(-0.04, 0.5, NS.DEPTH_LABEL[d], transform=ax.transAxes, rotation=90,
                    va='center', ha='right', fontsize=7)
        NS.panel_letter(ax, 'abcdefghijkl'[k], x=0.0, y=0.93)
        k += 1
fig.suptitle(f'Quantile regression forest · {level}% prediction interval',
             x=0.13, y=0.935, ha='left', fontsize=7.5, fontweight='bold')
NS.save(fig, os.path.join(C.FIG_DIR, 'Fig8_QRF_prediction_interval_maps'))
plt.show()
